# Recurrent Charge — Target Merchant Genérico — SeqNN v1.9.0

## 0. Setup

Entorno, imports y semillas globales.

In [1]:
%pip install -q polars torch scikit-learn scipy joblib google-cloud-bigquery pyarrow

Note: you may need to restart the kernel to use updated packages.


In [2]:
# 0.2 — IMPORTS + SEMILLAS GLOBALES
import os, time, json, gc, pickle, random
from pathlib import Path
from datetime import timedelta, datetime, date
from typing import Dict, List, Tuple, Optional, NamedTuple
from dataclasses import dataclass

import numpy as np
import polars as pl
import pandas as pd
import joblib

import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

from google.cloud import bigquery
from sklearn.preprocessing import StandardScaler

import warnings
warnings.filterwarnings('ignore')

# Semillas globales: reproducibilidad de subsample, init de pesos y shuffles.
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

print('Imports OK')
print(f'  PyTorch: {torch.__version__}')
print(f'  Device:  {"cuda" if torch.cuda.is_available() else "cpu"}')
print(f'  Seed:    {SEED}')

Imports OK
  PyTorch: 2.10.0+cu128
  Device:  cuda
  Seed:    42


## 1. CONFIG

Unico diccionario de configuracion. Se removieron `HMM_FEATURES`, `REQUIRED_COLUMNS` y `_LEGACY_REQUIRED` (codigo muerto: `HMM_FEATURES` se construye en la Seccion 5; el query siempre es `SELECT *`).

In [3]:
CONFIG = {
    #  BigQuery
    'project': 'spin-aip-singularity-data-sb',
    'dataset': 'recurrent_charger_netflix_merchant',
    'table': 'recurrent_charger_google_one_merchant_exp_v5_0_1',  # <- unica linea a cambiar para correr otro merchant

    #  Splits temporales 
    'train_pct': 0.80,
    'val_pct': 0.10,
    'test_pct': 0.10,

    # ═══ Subsample ═══
    'subsample_users': None,
    'stratify_col': 'decline_rate_30d',
    'stratify_bins': 5,

    # ═══ Limpieza ═══
    'clip_zscore': 5.0,
    'min_sequence_length': 3,

    # ═══ Imputación ═══
    'impute_days_since_approval': 27,
    'impute_days_to_due': -4,
    'impute_payment_interval_avg': 30,
    'impute_payment_interval_std': 9,

    # ═══ Caps ═══
    'cap_days_since': 180,
    'cap_days_to_due_min': -60,
    'cap_days_to_due_max': 60,

    # ═══ Backtesting ═══
    'backtest_horizon_days': 45,
    'baseline_interval_days': 30,

    # ═══ NN ESPECÍFICO ═══
    'objective_mode': 'next_attempt',
    'lookback_window': 8,

    'bucket_edges': [0, 1, 2, 3, 4, 5, 7, 14, 30],
    'n_time_buckets': 9,   
    'bucket_centers': [45.0, 0.5, 1.5, 2.5, 3.5, 4.5, 6.0, 10.5, 22.0, 37.5],

    # FILTRADO DE RETRIES
    'filter_retry_gap_days': 1,
    
    # Modelo
    'models_to_train': ['GRU', 'RNN', 'LSTM', 'TCN', 'DEEP_GRU'],

    'model_configs': {
    'GRU':  {'hidden_size': 256, 'num_layers': 4, 'dropout': 0.4},
    'RNN':  {'hidden_size': 256, 'num_layers': 4, 'dropout': 0.4},
    'LSTM': {'hidden_size': 256, 'num_layers': 4, 'dropout': 0.4},
    'TCN':  {'num_channels': [64, 128, 256 ,128], 'kernel_size': 4, 'dropout': 0.1},
    'DEEP_GRU': {'hidden_size': 256, 'num_layers': 4, 'dropout': 0.4, 'drop_path': 0.1},  
    },
    'start_date': '2024-01-01',
    'end_date':   '2026-05-01',
    'batch_size': 2048,
    'learning_rate': 1e-3,
    'max_epochs': 100,
    'patience': 5,
    'n_seeds': 1,  
    'weight_decay': 1e-5,
    'use_multitask': True,
    'multitask_weight': 0.3,
    'ordinal_loss_weight': 0.5,
    'focal_gamma': 2.0,
    
# Focal Loss + Class Weights (accuracy boost)
    'use_focal_loss': True,             # switch maestro; antes implícito=False
    'use_class_weights': True,          # pondera por frecuencia inversa
    'class_weight_power': 0.85,          # 0=off, 0.5=sqrt-inverse, 1=full-inverse
    'label_smoothing': 0.1,             # ordinal-aware: suaviza buckets adyacentes
    'grad_clip_max_norm': 1.0,          # evita gradient explosion en redes profundas
    # FIX 3: PENALIZACIÓN ASIMÉTRICA
    'late_penalty_alpha': 2.0,

    # FIX 6: REGRESSION HEAD
    'regression_weight': 0.5,
    'regression_weight_deep_gru': 0.05,  # vs 0.5 actual en otros modelos
    'regression_blend_alpha': 0.3,  # NOTA: no usado en pipeline actual (solo bucket head via quantile decoder)

    # GO/NO-GO
    'gate_mae_threshold': 5.0,
    'gate_vs_baseline_ratio': 0.85,
    'gate_p90_threshold': 12.0,

    # Action segments
    'action_segments': {
        'immediate':   {'days_min': 0,  'days_max': 7,   'label': '≤7d'},
        'short_term':  {'days_min': 8,  'days_max': 14,  'label': '8-14d'},
        'medium_term': {'days_min': 15, 'days_max': 30,  'label': '15-30d'},
        'long_term':   {'days_min': 31, 'days_max': 999, 'label': '>30d/no-event'},
    },

    'output_dir': 'artifacts_v1_9_0',
}

OUTPUT_DIR = Path(CONFIG['output_dir'])
OUTPUT_DIR.mkdir(exist_ok=True, parents=True)
print(f"   Buckets: {CONFIG['n_time_buckets']+1} (adaptativos)")
print(f"   GRU: hidden={CONFIG['model_configs']['GRU']['hidden_size']}, "
      f"layers={CONFIG['model_configs']['GRU']['num_layers']}")
print(f"   Late penalty α: {CONFIG['late_penalty_alpha']}")
print(f"   Regression blend: {CONFIG['regression_blend_alpha']}")
print(f"   Retry filter: ≤{CONFIG['filter_retry_gap_days']}d")
for mt, mc in CONFIG['model_configs'].items():
    print(f"   {mt}: {mc}")

   Buckets: 10 (adaptativos)
   GRU: hidden=256, layers=4
   Late penalty α: 2.0
   Regression blend: 0.3
   Retry filter: ≤1d
   GRU: {'hidden_size': 256, 'num_layers': 4, 'dropout': 0.4}
   RNN: {'hidden_size': 256, 'num_layers': 4, 'dropout': 0.4}
   LSTM: {'hidden_size': 256, 'num_layers': 4, 'dropout': 0.4}
   TCN: {'num_channels': [64, 128, 256, 128], 'kernel_size': 4, 'dropout': 0.1}
   DEEP_GRU: {'hidden_size': 256, 'num_layers': 4, 'dropout': 0.4, 'drop_path': 0.1}


## 2. Data Pipeline

Carga desde BigQuery con cache local en parquet.

In [4]:
CACHE_PATH = "data/reminders_recurrent_charger_google_one_merchant_exp_v5_0_1_cache.parquet"

def load_data_from_bigquery(config: dict) -> pl.DataFrame:
    """Carga datos desde BigQuery a Polars DataFrame (optimizado)."""
    client = bigquery.Client(project=config['project'])

    # Sin ORDER BY — se ordena después en Polars
    query = f"""
    SELECT *
    FROM `{config['project']}.{config['dataset']}.{config['table']}`
    WHERE event_date BETWEEN '{config['start_date']}' AND '{config['end_date']}'
    """

    print(" Ejecutando query en BigQuery...")
    # Directo Arrow → Polars (sin paso intermedio por Pandas)
    arrow_table = client.query(query).to_arrow()
    df = pl.from_arrow(arrow_table)
    del arrow_table
    gc.collect()

    df = df.sort(["userIdentifier", "event_ts", "attempt_seq"])

    print(f" Datos cargados: {df.height:,} rows, {df.width} cols")
    print(f"   Memoria: {df.estimated_size('mb'):.1f} MB")
    print(f"   Usuarios: {df['userIdentifier'].n_unique():,}")
    print(f"   Rango temporal: {df['event_date'].min()} → {df['event_date'].max()}")

    return df


def load_data(config: dict) -> pl.DataFrame:
    """Carga con cache local en parquet para re-ejecuciones rápidas."""
    if os.path.exists(CACHE_PATH):
        print(f" Cargando desde cache local ({CACHE_PATH})...")
        df = pl.read_parquet(CACHE_PATH)
        print(f"   {df.height:,} rows, {df.estimated_size('mb'):.1f} MB")
        return df

    df = load_data_from_bigquery(config)

    os.makedirs(os.path.dirname(CACHE_PATH), exist_ok=True)
    df.write_parquet(CACHE_PATH)
    print(f" Cache guardado en {CACHE_PATH}")

    return df


df = load_data(CONFIG)

 Ejecutando query en BigQuery...
 Datos cargados: 12,639,315 rows, 112 cols
   Memoria: 13158.2 MB
   Usuarios: 601,417
   Rango temporal: 2024-01-02 → 2026-05-01
 Cache guardado en data/reminders_recurrent_charger_google_one_merchant_exp_v5_0_1_cache.parquet


In [5]:
FORCE_REFRESH_CACHE = False   

if FORCE_REFRESH_CACHE and os.path.exists(CACHE_PATH):
    os.remove(CACHE_PATH)
    print(f"✅ Cache borrado: {CACHE_PATH}")
    print(f"   Cell 7 va a re-ejecutar la query a BigQuery con SELECT *")
elif os.path.exists(CACHE_PATH):
    print(f"⚠️  Cache existente conservado (FORCE_REFRESH_CACHE=False)")
else:
    print(f"   No hay cache previo, cell 7 hará query directa a BigQuery")

✅ Cache borrado: data/reminders_recurrent_charger_google_one_merchant_exp_v5_0_1_cache.parquet
   Cell 7 va a re-ejecutar la query a BigQuery con SELECT *


## 3. Filtrado de Retries Intraday + Deduplicacion

Colapsa sesiones de retry (gaps <= 1d) y deduplica PKs antes de cualquier split temporal.

In [5]:
def filter_retries(df: pl.DataFrame, config: dict) -> pl.DataFrame:
    """
    Colapsa sesiones de retry: mantiene solo el primer intento de cada
    sesión donde los gaps consecutivos son ≤ filter_retry_gap_days.

    Después de filtrar, re-calcula days_to_next_attempt para que apunte
    al siguiente intento NO-retry.

    Args:
        df: DataFrame con columns [userIdentifier, event_ts, days_since_prev_attempt, ...]
        config: debe contener 'filter_retry_gap_days'

    Returns:
        df_filtered: DataFrame sin retries, con targets recalculados
    """
    gap_threshold = config['filter_retry_gap_days']
    n_before = df.height

    print(f"\n🔧 FIX 4: Filtrando retries (gap ≤ {gap_threshold}d)...")

    # Asegurar orden
    df = df.sort(['userIdentifier', 'event_ts', 'attempt_seq'])

    # Marcar si es retry: gap al intento anterior ≤ threshold
    # El primer intento de cada usuario NO es retry (no tiene gap previo)
    df = df.with_columns(
        pl.when(
            pl.col('days_since_prev_attempt').is_not_null() &
            (pl.col('days_since_prev_attempt') <= gap_threshold)
        ).then(pl.lit(True))
        .otherwise(pl.lit(False))
        .alias('_is_retry')
    )

    # Asignar session_id: incrementa cuando NO es retry
    # Esto agrupa retries consecutivos con su intento inicial
    df = df.with_columns(
        (~pl.col('_is_retry')).cum_sum().over('userIdentifier').alias('_session_id')
    )

    # Mantener solo el PRIMER evento de cada sesión (el intento original)
    df_filtered = (
        df
        .with_columns(
            pl.col('event_ts').rank('ordinal').over(['userIdentifier', '_session_id']).alias('_rank')
        )
        .filter(pl.col('_rank') == 1)
        .drop(['_is_retry', '_session_id', '_rank'])
    )

    # Recalcular days_to_next_attempt post-filtrado
    # Ahora "next attempt" es el siguiente intento NO-retry
    df_filtered = df_filtered.sort(['userIdentifier', 'event_ts', 'attempt_seq'])
    df_filtered = df_filtered.with_columns(
        (
            pl.col('event_ts').shift(-1).over('userIdentifier') - pl.col('event_ts')
        ).dt.total_days().cast(pl.Float64).alias('days_to_next_attempt_clean')
    )

    # Reemplazar days_to_next_attempt con la versión limpia
    # El último evento de cada usuario queda como null (censurado)
    df_filtered = df_filtered.with_columns(
        pl.col('days_to_next_attempt_clean').alias('days_to_next_attempt')
    ).drop('days_to_next_attempt_clean')

    # También recalcular days_since_prev_attempt
    df_filtered = df_filtered.with_columns(
        (
            pl.col('event_ts') - pl.col('event_ts').shift(1).over('userIdentifier')
        ).dt.total_days().cast(pl.Float64).alias('days_since_prev_attempt')
    )

    n_after = df_filtered.height
    n_removed = n_before - n_after
    pct_removed = n_removed / n_before * 100

    print(f"   Antes: {n_before:,} eventos")
    print(f"   Retries eliminados: {n_removed:,} ({pct_removed:.1f}%)")
    print(f"   Después: {n_after:,} eventos")

    # Verificar distribución post-filtrado
    tte_clean = df_filtered.filter(
        pl.col('days_to_next_attempt').is_not_null()
    )['days_to_next_attempt']
    if len(tte_clean) > 0:
        print(f"   TTE post-filtrado: median={tte_clean.median():.1f}d, "
              f"mean={tte_clean.mean():.1f}d, "
              f"p10={tte_clean.quantile(0.1):.0f}d, "
              f"p90={tte_clean.quantile(0.9):.0f}d")

    return df_filtered



# ═══ EJECUTAR ═══
df = filter_retries(df, CONFIG)


🔧 FIX 4: Filtrando retries (gap ≤ 1d)...
   Antes: 12,639,315 eventos
   Retries eliminados: 6,288,059 (49.7%)
   Después: 6,351,256 eventos
   TTE post-filtrado: median=9.0d, mean=18.5d, p10=2d, p90=31d


In [6]:
# ═══════════════════════════════════════════════════════════════════════════════
# DEDUPLICACIÓN — FIX v1.5.6: Movido ANTES de splits/normalización/sequences
# Ejecuta dedup sobre df global ANTES de cualquier split temporal.
# Esto garantiza que train/val/test nunca contengan duplicados.
# ═══════════════════════════════════════════════════════════════════════════════

pk_cols = ['userIdentifier', 'event_ts', 'attempt_seq']
n_before_dedup = df.height
n_unique_pk = df.select(pk_cols).unique().height
n_dupes = n_before_dedup - n_unique_pk

print(f"\n🔧 DEDUP v1.5.6: Verificando PKs ({', '.join(pk_cols)})")
print(f"   Total rows: {n_before_dedup:,}")
print(f"   PKs únicos: {n_unique_pk:,}")
print(f"   Duplicados: {n_dupes:,} ({n_dupes/n_before_dedup*100:.2f}%)")

if n_dupes > 0:
    df = df.unique(
        subset=pk_cols,
        keep='first',
        maintain_order=True,
    )
    print(f"   ✅ Dedup aplicado: {n_before_dedup:,} → {df.height:,} rows")
else:
    print(f"   ✅ Sin duplicados — df intacto")

# Verificación post-dedup
assert df.height == df.select(pk_cols).unique().height, "DEDUP FALLÓ"
print(f"   ✅ Verificación post-dedup: 0 duplicados")



🔧 DEDUP v1.5.6: Verificando PKs (userIdentifier, event_ts, attempt_seq)
   Total rows: 6,351,256
   PKs únicos: 6,208,658
   Duplicados: 142,598 (2.25%)
   ✅ Dedup aplicado: 6,351,256 → 6,208,658 rows
   ✅ Verificación post-dedup: 0 duplicados


## 4. Target Alignment

Alinea `time_to_event_days` y `event_observed` segun `objective_mode`.

In [7]:
# ═══════════════════════════════════════════════════════════════════════════════
# TARGET ALIGNMENT — objective_mode controls what we predict
# ═══════════════════════════════════════════════════════════════════════════════

def align_targets(df: pl.DataFrame, config: dict) -> pl.DataFrame:
    """
    Alinea time_to_event_days y event_observed según objective_mode.
    
    Si objective_mode == 'next_attempt':
        time_to_event_days := days_to_next_attempt  (gap time entre intentos)
        event_observed := days_to_next_attempt IS NOT NULL
    Si objective_mode == 'next_approval':
        time_to_event_days := days_to_next_approval
        event_observed := days_to_next_approval IS NOT NULL
    """
    mode = config['objective_mode']
    print(f"🎯 Alineando targets → objective_mode = '{mode}'")
    
    if mode == 'next_attempt':
        # El SQL v5.0 ya define time_to_event_days = days_to_next_attempt
        # Pero re-derivamos explícitamente para estar 100% seguros
        if 'days_to_next_attempt' in df.columns:
            df = df.with_columns([
                pl.col('days_to_next_attempt').alias('time_to_event_days'),
                pl.col('days_to_next_attempt').is_not_null().cast(pl.Int8).alias('event_observed'),
                pl.col('days_to_next_attempt').is_null().cast(pl.Int8).alias('is_censored'),
            ])
            print("   ✅ time_to_event_days := days_to_next_attempt")
        else:
            print("   ⚠️  days_to_next_attempt no encontrado, usando time_to_event_days existente")
    
    elif mode == 'next_approval':
        if 'days_to_next_approval' in df.columns:
            df = df.with_columns([
                pl.col('days_to_next_approval').alias('time_to_event_days'),
                pl.col('days_to_next_approval').is_not_null().cast(pl.Int8).alias('event_observed'),
                pl.col('days_to_next_approval').is_null().cast(pl.Int8).alias('is_censored'),
            ])
            print("   ✅ time_to_event_days := days_to_next_approval")
        else:
            raise ValueError("days_to_next_approval no existe en dataset")
    else:
        raise ValueError(f"objective_mode '{mode}' no reconocido. Usa 'next_attempt' o 'next_approval'")
    
    # Binary survival targets (re-derive from aligned time_to_event_days)
    for window_days, col_name in [(3, 'y_3d'), (7, 'y_7d'), (14, 'y_14d'), (30, 'y_30d')]:
        df = df.with_columns(
            pl.when(pl.col('time_to_event_days').is_not_null() & (pl.col('time_to_event_days') <= window_days))
              .then(pl.lit(1))
              .otherwise(pl.lit(0))
              .cast(pl.Float32)
              .alias(col_name)
        )
    
    # Stats
    n_observed = df.filter(pl.col('event_observed') == 1).height
    n_censored = df.filter(pl.col('is_censored') == 1).height
    tte_stats = df.filter(pl.col('event_observed') == 1)['time_to_event_days']
    
    print(f"   Eventos observados: {n_observed:,} ({n_observed/df.height*100:.1f}%)")
    print(f"   Censurados: {n_censored:,} ({n_censored/df.height*100:.1f}%)")
    if len(tte_stats) > 0:
        print(f"   TTE (observados): median={tte_stats.median():.1f}d, "
              f"mean={tte_stats.mean():.1f}d, p90={tte_stats.quantile(0.9):.1f}d")
    
    return df


df = align_targets(df, CONFIG)

🎯 Alineando targets → objective_mode = 'next_attempt'
   ✅ time_to_event_days := days_to_next_attempt
   Eventos observados: 5,618,296 (90.5%)
   Censurados: 590,362 (9.5%)
   TTE (observados): median=9.0d, mean=18.6d, p90=31.0d


## 5. Feature Engineering

**Consolidado (fix P1-1).** Antes `HMM_FEATURES` se definia en CONFIG y se sobrescribia en otra celda. Aqui se construye **una sola vez**: baseline + expansion, devuelto explicitamente. Si una columna fuente falta, la validacion final aborta con un mensaje claro (no rompe en silencio).

In [8]:
# BUILD FEATURES

def build_hmm_features_v2(df: pl.DataFrame, config: dict) -> pl.DataFrame:
    """Construye features"""
    print("⚙️  Building features v2...")
    
    df = df.with_columns([
        # Log transforms
        pl.col('attempts_7d').fill_null(0).add(1).log().alias('log_attempts_7d'),
        pl.col('attempts_30d').fill_null(0).add(1).log().alias('log_attempts_30d'),
        pl.col('rejects_7d').fill_null(0).add(1).log().alias('log_rejects_7d'),
        (pl.col('attempts_7d').fill_null(0) + 1).log().alias('log_attempts_7d_new'),
        (pl.col('rejects_7d').fill_null(0) + 1).log().alias('log_rejects_7d_new'),
        (pl.col('rejects_7d').fill_null(0) / (pl.col('attempts_7d').fill_null(0) + 1)).alias('retry_intensity_7d_norm'),

        # Decline rate
        pl.col('decline_rate_30d').fill_null(0).clip(0, 1).alias('decline_rate_30d_clipped'),
        
        # Days since con imputación y cap
        pl.when(pl.col('days_since_last_attempt').is_null())
          .then(config['cap_days_since'])
          .otherwise(pl.col('days_since_last_attempt'))
          .clip(0, config['cap_days_since'])
          .alias('days_since_last_attempt_hmm'),
        
        pl.when(pl.col('days_since_last_reject').is_null())
          .then(config['cap_days_since'])
          .otherwise(pl.col('days_since_last_reject'))
          .clip(0, config['cap_days_since'])
          .alias('days_since_last_reject_hmm'),
        
        pl.when(pl.col('days_since_last_approval').is_null())
          .then(pl.lit(1))
          .otherwise(pl.lit(0))
          .alias('missing_last_approval'),
        
        pl.when(pl.col('days_since_last_approval').is_null())
          .then(config['impute_days_since_approval'])
          .otherwise(pl.col('days_since_last_approval'))
          .clip(0, config['cap_days_since'])
          .alias('days_since_last_approval_hmm'),
        
    #  Days to due 
        pl.col('days_to_predicted_due').is_null()
          .cast(pl.Int8)
          .alias('missing_predicted_due'),          # NUEVO: indicator

        pl.when(pl.col('days_to_predicted_due').is_null())
          .then(config['impute_days_to_due'])       # ahora = -4 (mediana), no 60
          .otherwise(pl.col('days_to_predicted_due'))
          .clip(config['cap_days_to_due_min'], config['cap_days_to_due_max'])
          .alias('days_to_predicted_due_hmm'),

        #  Payment interval stddev 
        pl.col('payment_interval_stddev').is_null()
          .cast(pl.Int8)
          .alias('missing_interval_std'),            

        pl.when(pl.col('payment_interval_stddev').is_null())
          .then(config['impute_payment_interval_std'])  
          .otherwise(pl.col('payment_interval_stddev'))
          .alias('payment_interval_stddev_hmm'),

        # ═══ Net flow ratio 
        (pl.col('net_flow_30d').fill_null(0) /
         pl.when(pl.col('outflow_30d').fill_null(0).abs() < 1.0)   
           .then(pl.lit(1.0))                                       
           .otherwise(pl.col('outflow_30d').fill_null(0).abs())
        ).clip(-10, 10).alias('net_flow_30d_ratio'),                
        
        # Liquidity pressure
        pl.when(pl.col('inflow_30d').fill_null(0) < 0.01)
          .then(pl.lit(1))
          .otherwise(pl.lit(0))
          .alias('no_inflow_30d'),
        
        (pl.col('outflow_30d').fill_null(0).abs() / 
         pl.when(pl.col('inflow_30d').fill_null(0) < 0.01)
           .then(pl.lit(0.01))
           .otherwise(pl.col('inflow_30d').fill_null(0))
        ).clip(0, 10).alias('liquidity_pressure'),
    ])
    
    print("✅ Features baseline construidas")
    return df


df = build_hmm_features_v2(df, CONFIG)

# v1.8.2 — EXPANSIÓN DE FEATURES (autosuficiente, llama a baseline primero)
import polars as pl
import numpy as np

# PRIMERO: garantizar que las features baseline existen en df
# Si no están, llamar a build_hmm_features_v2 que las construye
if 'log_attempts_7d' not in df.columns:
    print("⚙️  Features baseline no encontradas, ejecutando build_hmm_features_v2 primero...")
    df = build_hmm_features_v2(df, CONFIG)
    print("   ✓ Baseline construida")

# RESET defensivo de HMM_FEATURES
HMM_FEATURES_BASELINE = [
    'log_attempts_7d', 'log_rejects_7d', 'decline_rate_30d_clipped',
    'days_since_last_attempt_hmm', 'days_since_last_reject_hmm',
    'days_since_last_approval_hmm', 'days_to_predicted_due_hmm',
    'payment_interval_stddev_hmm', 'net_flow_30d_ratio',
    'liquidity_pressure', 'no_inflow_30d',
    'missing_last_approval', 'missing_predicted_due', 'missing_interval_std',
    'log_attempts_7d_new', 'retry_intensity_7d_norm',
]

# Helper: log con signo para montos que pueden ser negativos
def signed_log_expr(col_name, alias):
    col = pl.col(col_name).fill_null(0).cast(pl.Float64)
    return (col.sign() * (col.abs() + 1).log()).alias(alias)

print("⚙️  Construyendo features expandidas (todos los bloques)...")

# Idempotencia a nivel de DataFrame
new_cols_already_present = [c for c in [
    'log_target_txn_m1', 'log_inflow_7d', 'attempts_in_day_clipped',
    'log_amount_mxn', 'days_on_book_clipped'
] if c in df.columns]

if len(new_cols_already_present) == 5:
    print("⚠️  Features expandidas ya existen en df. Saltando construcción.")
else:
    new_feature_exprs = []

    # === BLOCK_TARGET_CYCLE (9 features) ===
    new_feature_exprs.extend([
        (pl.col('target_txn_m1').fill_null(0) + 1).log().alias('log_target_txn_m1'),
        (pl.col('target_txn_m2').fill_null(0) + 1).log().alias('log_target_txn_m2'),
        (pl.col('target_txn_m3').fill_null(0) + 1).log().alias('log_target_txn_m3'),
        (pl.col('target_txn_total_3m').fill_null(0) + 1).log().alias('log_target_txn_total_3m'),
        pl.col('target_months_active').fill_null(0).clip(0, 24).cast(pl.Float64).alias('target_months_active_clipped'),
        (pl.col('days_since_last_payment').fill_null(365) + 1).log().alias('log_days_since_last_payment'),
        pl.col('days_since_last_payment').is_null().cast(pl.Int8).alias('missing_days_since_payment'),
        (2 * np.pi * pl.col('billing_day_of_month').fill_null(15).cast(pl.Float64) / 31).sin().alias('billing_day_sin'),
        (2 * np.pi * pl.col('billing_day_of_month').fill_null(15).cast(pl.Float64) / 31).cos().alias('billing_day_cos'),
    ])

    # === BLOCK_LIQUIDITY_SHORT (9 features) ===
    new_feature_exprs.extend([
        (pl.col('inflow_7d').fill_null(0).abs() + 1).log().alias('log_inflow_7d'),
        (pl.col('outflow_7d').fill_null(0).abs() + 1).log().alias('log_outflow_7d'),
        signed_log_expr('net_flow_7d', 'log_signed_net_flow_7d'),
        (pl.col('inflow_14d').fill_null(0).abs() + 1).log().alias('log_inflow_14d'),
        (pl.col('outflow_14d').fill_null(0).abs() + 1).log().alias('log_outflow_14d'),
        signed_log_expr('net_flow_14d', 'log_signed_net_flow_14d'),
        (pl.col('inflow_30d').fill_null(0).abs() + 1).log().alias('log_inflow_30d'),
        (pl.col('outflow_30d').fill_null(0).abs() + 1).log().alias('log_outflow_30d'),
        signed_log_expr('net_flow_30d', 'log_signed_net_flow_30d'),
    ])

    # === BLOCK_RETRY_FINE (7 features) ===
    new_feature_exprs.extend([
        pl.col('attempts_in_day').fill_null(0).clip(0, 20).cast(pl.Float64).alias('attempts_in_day_clipped'),
        pl.col('rejects_in_day').fill_null(0).clip(0, 20).cast(pl.Float64).alias('rejects_in_day_clipped'),
        pl.col('approvals_in_day').fill_null(0).clip(0, 20).cast(pl.Float64).alias('approvals_in_day_clipped'),
        (pl.col('attempts_14d').fill_null(0) + 1).log().alias('log_attempts_14d'),
        (pl.col('rejects_14d').fill_null(0) + 1).log().alias('log_rejects_14d'),
        (pl.col('approvals_14d').fill_null(0) + 1).log().alias('log_approvals_14d'),
        (pl.col('approvals_30d').fill_null(0) + 1).log().alias('log_approvals_30d'),
    ])

    # === BLOCK_AMOUNTS (4 features) ===
    new_feature_exprs.extend([
        (pl.col('amount_mxn').fill_null(0) + 1).log().alias('log_amount_mxn'),
        pl.col('amount_mxn').is_null().cast(pl.Int8).alias('missing_amount_mxn'),
        (pl.col('plan_amount').fill_null(0) + 1).log().alias('log_plan_amount'),
        pl.col('payment_interval_avg').fill_null(30).clip(0, 90).cast(pl.Float64).alias('payment_interval_avg_clipped'),
    ])

    # === BLOCK_DEMOGRAPHIC_SAFE (2 features) ===
    new_feature_exprs.extend([
        pl.col('days_on_book').fill_null(0).clip(0, 5000).cast(pl.Float64).alias('days_on_book_clipped'),
        pl.col('user_age_years').fill_null(35).clip(18, 90).cast(pl.Float64).alias('user_age_years_clipped'),
    ])

    df = df.with_columns(new_feature_exprs)
    print(f"✅ Features expandidas: {len(new_feature_exprs)} columnas agregadas a df")

# Lista canónica de features nuevas
NEW_FEATURE_NAMES = [
    'log_target_txn_m1', 'log_target_txn_m2', 'log_target_txn_m3', 'log_target_txn_total_3m',
    'target_months_active_clipped', 'log_days_since_last_payment',
    'missing_days_since_payment', 'billing_day_sin', 'billing_day_cos',
    'log_inflow_7d', 'log_outflow_7d', 'log_signed_net_flow_7d',
    'log_inflow_14d', 'log_outflow_14d', 'log_signed_net_flow_14d',
    'log_inflow_30d', 'log_outflow_30d', 'log_signed_net_flow_30d',
    'attempts_in_day_clipped', 'rejects_in_day_clipped', 'approvals_in_day_clipped',
    'log_attempts_14d', 'log_rejects_14d', 'log_approvals_14d', 'log_approvals_30d',
    'log_amount_mxn', 'missing_amount_mxn', 'log_plan_amount', 'payment_interval_avg_clipped',
    'days_on_book_clipped', 'user_age_years_clipped',
]

# RESET de HMM_FEATURES desde baseline + nuevas (idempotente)
HMM_FEATURES = HMM_FEATURES_BASELINE + NEW_FEATURE_NAMES

# Validación: confirmar que toda feature existe en df
missing = [f for f in HMM_FEATURES if f not in df.columns]
if missing:
    raise ValueError(f"❌ Features faltantes en df: {missing}")

print(f"\n✅ HMM_FEATURES total: {len(HMM_FEATURES)} ({len(HMM_FEATURES_BASELINE)} baseline + {len(NEW_FEATURE_NAMES)} nuevas)")
print(f"   Validación: todas las features existen en df ✓")

⚙️  Building features v2...
✅ Features baseline construidas
⚙️  Construyendo features expandidas (todos los bloques)...
✅ Features expandidas: 31 columnas agregadas a df

✅ HMM_FEATURES total: 47 (16 baseline + 31 nuevas)
   Validación: todas las features existen en df ✓


## 6. Temporal Splits

Split por semanas completas, sin overlap temporal. Proporciones tomadas de CONFIG (`train/val/test_pct`).

In [9]:
# TEMPORAL SPLITS (segun CONFIG: 80/10/10) — EVENT-LEVEL

def create_temporal_splits(df: pl.DataFrame, config: dict) -> Tuple[pl.DataFrame, pl.DataFrame, pl.DataFrame]:
    """Split temporal por event_ts, agrupado en semanas completas."""
    
    # Derivar week_start SOLO para agrupamiento de split
    df = df.with_columns(
        pl.col('event_date').cast(pl.Date).dt.truncate('1w').alias('_split_week')
    )
    
    # Ordenar por event_ts (NO por week_start)
    df = df.sort(['event_ts', 'attempt_seq'])
    
    # Split por semanas completas
    weeks_sorted = df['_split_week'].unique().sort()
    n_weeks = len(weeks_sorted)
    
    train_end_idx = int(n_weeks * config['train_pct'])
    val_end_idx = int(n_weeks * (config['train_pct'] + config['val_pct']))
    
    train_weeks = weeks_sorted[:train_end_idx]
    val_weeks = weeks_sorted[train_end_idx:val_end_idx]
    test_weeks = weeks_sorted[val_end_idx:]
    
    df_train = df.filter(pl.col('_split_week').is_in(train_weeks))
    df_val = df.filter(pl.col('_split_week').is_in(val_weeks))
    df_test = df.filter(pl.col('_split_week').is_in(test_weeks))
    
    # Validar no-overlap temporal
    train_max = df_train['event_ts'].max()
    val_min = df_val['event_ts'].min()
    val_max = df_val['event_ts'].max()
    test_min = df_test['event_ts'].min()
    
    print(f"\n📊 Splits temporales (event-level):")
    print(f"   Train: {df_train.height:,} events, {len(train_weeks)} weeks")
    print(f"          {df_train['event_date'].min()} → {df_train['event_date'].max()}")
    print(f"   Val:   {df_val.height:,} events, {len(val_weeks)} weeks")
    print(f"          {df_val['event_date'].min()} → {df_val['event_date'].max()}")
    print(f"   Test:  {df_test.height:,} events, {len(test_weeks)} weeks")
    print(f"          {df_test['event_date'].min()} → {df_test['event_date'].max()}")
    
    # Validar no-leak
    assert train_max < val_min, f"❌ LEAK: train max {train_max} >= val min {val_min}"
    assert val_max < test_min, f"❌ LEAK: val max {val_max} >= test min {test_min}"
    print(f"   ✅ No temporal overlap between splits")
    
    # Drop helper column
    df_train = df_train.drop('_split_week')
    df_val = df_val.drop('_split_week')
    df_test = df_test.drop('_split_week')
    
    return df_train, df_val, df_test


df_train, df_val, df_test = create_temporal_splits(df, CONFIG)


📊 Splits temporales (event-level):
   Train: 4,107,602 events, 97 weeks
          2024-01-02 → 2025-11-09
   Val:   1,016,682 events, 12 weeks
          2025-11-10 → 2026-02-01
   Test:  1,084,374 events, 13 weeks
          2026-02-02 → 2026-05-01
   ✅ No temporal overlap between splits


## 7. Normalizacion + Persistencia del Scaler

`StandardScaler` ajustado SOLO en train. Se persisten scaler y manifest de features (necesarios para inferencia de kernel fresco).

In [10]:
# NORMALIZATION (StandardScaler + clip z-score ±5)

def normalize_features(
    df_train: pl.DataFrame,
    df_val: pl.DataFrame,
    df_test: pl.DataFrame,
    feature_cols: List[str],
    config: dict
) -> Tuple[pl.DataFrame, pl.DataFrame, pl.DataFrame, StandardScaler]:
    """Normalización - fit SOLO en train + clip."""
    print("🔧 Normalizando features...")
    
    # Fit scaler SOLO en train
    X_train = df_train.select(feature_cols).fill_null(0).to_numpy().astype(np.float32)
    scaler = StandardScaler()
    scaler.fit(X_train)
    
    # Transform each split
    datasets = {'train': df_train, 'val': df_val, 'test': df_test}
    results = {}
    
    for name, df_split in datasets.items():
        X = df_split.select(feature_cols).fill_null(0).to_numpy().astype(np.float32)
        X_norm = scaler.transform(X)
        X_norm = np.clip(X_norm, -config['clip_zscore'], config['clip_zscore'])
        
        for i, col in enumerate(feature_cols):
            df_split = df_split.with_columns(
                pl.Series(f'{col}_norm', X_norm[:, i].astype(np.float32))
            )
        results[name] = df_split
    
    print(f"✅ Normalización completa (clip ±{config['clip_zscore']})")
    return results['train'], results['val'], results['test'], scaler


df_train, df_val, df_test, scaler = normalize_features(
    df_train, df_val, df_test, HMM_FEATURES, CONFIG
)

🔧 Normalizando features...
✅ Normalización completa (clip ±5.0)


In [11]:
# SCALER PERSISTENCE — v1.7.3 FIX P1-3
# Gap: scaler vivia solo en RAM. Inferencia con kernel limpio imposible.
import joblib, json as _json
_sp = OUTPUT_DIR / 'scaler_main.joblib'
_mp = OUTPUT_DIR / 'feature_manifest.json'
joblib.dump(scaler, _sp)
with open(_mp, 'w') as _f:
    _json.dump({
        'hmm_features': HMM_FEATURES,
        'clip_zscore': CONFIG['clip_zscore'],
        'scaler_mean': scaler.mean_.tolist(),
        'scaler_scale': scaler.scale_.tolist(),
    }, _f, indent=2)
print(f"  Scaler: {_sp}")
print(f"  Manifest: {_mp}")


  Scaler: artifacts_v1_9_0/scaler_main.joblib
  Manifest: artifacts_v1_9_0/feature_manifest.json


## 8. Validation Checks (V1-V5)

Gates de integridad de datos: divergencia de labels, sanity de intervalos, unicidad de PKs, leakage temporal.

In [12]:
# VALIDATION CHECKS V1–V5

validation_results = {}

# V1: Label check — next_attempt vs next_approval divergence
print("V1) Label: next_attempt vs next_approval divergence")
print("─" * 60)

if 'days_to_next_attempt' in df.columns and 'days_to_next_approval' in df.columns:
    df_v1 = df.filter(
        pl.col('days_to_next_attempt').is_not_null() & 
        pl.col('days_to_next_approval').is_not_null()
    )
    
    if df_v1.height > 0:
        divergent = df_v1.filter(
            pl.col('days_to_next_attempt') < pl.col('days_to_next_approval')
        )
        pct_divergent = divergent.height / df_v1.height * 100
        
        print(f"   Rows con ambos targets: {df_v1.height:,}")
        print(f"   Rows donde next_attempt < next_approval: {divergent.height:,} ({pct_divergent:.1f}%)")
        print(f"   → Estos son casos con RECHAZOS entre evento y siguiente aprobación")
        
        if pct_divergent > 5:
            print(f"   ⚠️  ADVERTENCIA: {pct_divergent:.1f}% divergencia — next_approval NO sirve para reminders")
        else:
            print(f"   ✅ Divergencia baja ({pct_divergent:.1f}%)")
        
        validation_results['v1_divergence_pct'] = pct_divergent
    else:
        print("   ⚠️  No hay rows con ambos targets para comparar")
        validation_results['v1_divergence_pct'] = None
else:
    print("   ⚠️  Columnas days_to_next_attempt/days_to_next_approval no encontradas")
    validation_results['v1_divergence_pct'] = None

# ─────────────────────────────────────────────────────────────────────────────
# V2: Sanity intervals — distribution check
# ─────────────────────────────────────────────────────────────────────────────
print(f"\nV2) Sanity intervals — distribución de gap times")
print("─" * 60)

for col_name in ['days_since_prev_attempt', 'days_to_next_attempt']:
    if col_name in df.columns:
        vals = df.filter(pl.col(col_name).is_not_null())[col_name]
        if len(vals) > 0:
            n_zero = (vals == 0).sum()
            n_one = (vals == 1).sum()
            pct_01 = (n_zero + n_one) / len(vals) * 100
            
            print(f"   {col_name}:")
            print(f"     N={len(vals):,}, median={vals.median():.1f}, mean={vals.mean():.1f}")
            print(f"     p10={vals.quantile(0.1):.0f}, p90={vals.quantile(0.9):.0f}")
            print(f"     0-day: {n_zero:,}, 1-day: {n_one:,} → {pct_01:.1f}% of total")
            
            if pct_01 > 30:
                print(f"     🔴 FLAG: {pct_01:.1f}% en 0-1 días — posible duplicación o retry ingest")
            elif pct_01 > 15:
                print(f"     🟡 FLAG: {pct_01:.1f}% en 0-1 días — revisar retry patterns")
            else:
                print(f"     ✅ Distribución razonable")
            
            validation_results[f'v2_{col_name}_pct_01'] = pct_01

# ─────────────────────────────────────────────────────────────────────────────
# V3: PK uniqueness — (userIdentifier, event_ts, attempt_seq)
# FIX v1.5.6: Dedup ya se ejecutó en celda 11 (antes de splits).
# Aquí solo verificamos que no hay duplicados post-pipeline.
# ─────────────────────────────────────────────────────────────────────────────
print(f"\nV3) PK uniqueness — VERIFICACIÓN (dedup ya aplicado)")
print("─" * 60)

pk_cols_v3 = ['userIdentifier', 'event_ts', 'attempt_seq']
n_total_v3 = df.height
n_unique_v3 = df.select(pk_cols_v3).unique().height
n_dupes_v3 = n_total_v3 - n_unique_v3
print(f"   Total rows: {n_total_v3:,}")
print(f"   Duplicados: {n_dupes_v3:,}")
assert n_dupes_v3 == 0, f"BLOCKER: {n_dupes_v3} duplicados detectados post-dedup"
print(f"   ✅ 0 duplicados — dedup confirmado")
validation_results['v3_pk_dupes'] = 0

# V4: Leakage temporal — features deben ser <= event_ts
# ─────────────────────────────────────────────────────────────────────────────
print(f"\nV4) Leakage temporal — features con 'last_' deben ser <= event_date")
print("─" * 60)

date_feature_cols = [c for c in df.columns if ('last_' in c and 'date' in c.lower()) 
                     and c not in ['xCard_last_four_digits']]
leakage_found = False

for col in date_feature_cols:
    if df[col].dtype in [pl.Date, pl.Datetime]:
        # Check if any feature date > event_date
        if df[col].dtype == pl.Date:
            n_leak = df.filter(pl.col(col) > pl.col('event_date')).height
        else:
            n_leak = df.filter(pl.col(col) > pl.col('event_ts')).height
        
        if n_leak > 0:
            print(f"   🔴 LEAKAGE: {col} > event_date en {n_leak:,} rows")
            leakage_found = True
        else:
            print(f"   ✅ {col} — no leakage")

if not leakage_found and len(date_feature_cols) > 0:
    print(f"   ✅ Ningún leakage detectado en {len(date_feature_cols)} columnas de fecha")
elif len(date_feature_cols) == 0:
    print(f"   ⚠️  No se encontraron columnas de fecha tipo 'last_*date*'")

validation_results['v4_leakage'] = leakage_found

# ─────────────────────────────────────────────────────────────────────────────
# V5: End-exclusive — rolling windows deben usar < end, no <=
# ─────────────────────────────────────────────────────────────────────────────
print(f"\nV5) End-exclusive — rolling windows")
print("─" * 60)
print("   ℹ️  Las features rolling del SQL v5.0 usan:")
print("      - BETWEEN DATE_SUB(d_txn, INTERVAL N-1 DAY) AND DATE_SUB(d_txn, INTERVAL 1 DAY)")
print("      - Esto excluye d_txn del propio evento (end-exclusive respecto al evento actual)")
print("   ✅ Validado en diseño del SQL. No aplica check programático aquí.")
validation_results['v5_end_exclusive'] = 'validated_by_design'

# ─────────────────────────────────────────────────────────────────────────────
# RESUMEN VALIDACIÓN
# ─────────────────────────────────────────────────────────────────────────────
print(f"\n{'='*60}")
print("RESUMEN VALIDACIÓN")
print(f"{'='*60}")

blockers = []
if validation_results.get('v3_pk_dupes', 0) > 0:
    blockers.append(f"V3: {validation_results['v3_pk_dupes']} PKs duplicados")
if validation_results.get('v4_leakage', False):
    blockers.append("V4: Leakage temporal detectado")

if blockers:
    print(f"\n🔴 BLOCKERS ({len(blockers)}):")
    for b in blockers:
        print(f"   - {b}")
    print(f"\n⛔ RESOLVER ANTES DE ENTRENAR")
else:
    print(f"\n✅ Todas las validaciones pasaron — safe to train")

V1) Label: next_attempt vs next_approval divergence
────────────────────────────────────────────────────────────
   Rows con ambos targets: 1,781,920
   Rows donde next_attempt < next_approval: 985,496 (55.3%)
   → Estos son casos con RECHAZOS entre evento y siguiente aprobación
   ⚠️  ADVERTENCIA: 55.3% divergencia — next_approval NO sirve para reminders

V2) Sanity intervals — distribución de gap times
────────────────────────────────────────────────────────────
   days_since_prev_attempt:
     N=5,607,241, median=9.0, mean=18.9
     p10=2, p90=31
     0-day: 0, 1-day: 136,860 → 2.4% of total
     ✅ Distribución razonable
   days_to_next_attempt:
     N=5,618,296, median=9.0, mean=18.6
     p10=2, p90=31
     0-day: 86,174, 1-day: 135,311 → 3.9% of total
     ✅ Distribución razonable

V3) PK uniqueness — VERIFICACIÓN (dedup ya aplicado)
────────────────────────────────────────────────────────────
   Total rows: 6,208,658
   Duplicados: 0
   ✅ 0 duplicados — dedup confirmado

V4) Leak

## 9. Bucket Encoding Adaptativo + Sequences

`tte_to_bucket_adaptive` es la **unica** bucketizacion del modelo (10 buckets). La bucketizacion de negocio (4 buckets) se define solo en el gate.

In [13]:
def tte_to_bucket_adaptive(tte: np.ndarray, config: dict) -> np.ndarray:
    """
    Convierte time-to-event (días) a bucket index usando intervalos adaptativos.

    Mapeo:
        null/NaN     → 0 (no-event/censored)
        [0, 1)       → 1
        [1, 2)       → 2
        [2, 3)       → 3
        [3, 4)       → 4
        [4, 5)       → 5
        [5, 7)       → 6
        [7, 14)      → 7
        [14, 30)     → 8
        [30, ∞)      → 9
        Fuera rango  → 0

    Args:
        tte: np.ndarray (N,) float — days to next event (NaN = censored)
        config: dict con 'bucket_edges'

    Returns:
        buckets: np.ndarray (N,) int64
    """
    edges = np.array(config['bucket_edges'], dtype=np.float64)
    n_event_buckets = config['n_time_buckets']  # 9

    buckets = np.zeros(len(tte), dtype=np.int64)  # default: 0 (no-event)

    valid = ~np.isnan(tte) & (tte >= 0)
    if valid.sum() > 0:
        # searchsorted con side='right': tte=0 → idx=1, tte=0.9 → idx=1,
        #   tte=1.0 → idx=2, etc.
        raw = np.searchsorted(edges, tte[valid], side='right').astype(np.int64)
        # Clamp: cualquier TTE ≥ último edge (30) → bucket 9
        raw = np.clip(raw, 1, n_event_buckets)
        buckets[valid] = raw

    return buckets


def print_bucket_distribution(buckets: np.ndarray, config: dict):
    """Muestra distribución de buckets con labels."""
    edges = config['bucket_edges']
    n_total = len(buckets)

    labels = ['no-event/censored']
    for i in range(len(edges)):
        if i < len(edges) - 1:
            labels.append(f'[{edges[i]}, {edges[i+1]})d')
        else:
            labels.append(f'[{edges[i]}, ∞)d')

    print("   Distribución buckets (adaptativo):")
    for b in range(config['n_time_buckets'] + 1):
        count = int((buckets == b).sum())
        pct = count / n_total * 100
        label = labels[b] if b < len(labels) else f'bucket-{b}'
        print(f"      bucket {b:2d} ({label:>20s}): {count:>10,} ({pct:5.1f}%)")


# ═════════════════════════════════════════════════════════════════════════════
# CELL 7 — SEQUENCE BUILDING (modificado para FIX 2)
# ═════════════════════════════════════════════════════════════════════════════

@dataclass
class SequenceOutput:
    """Salida de build_sequences(). Misma interfaz que v1.4.5."""
    X: np.ndarray                      # (N, lookback, n_features)
    y_bucket: np.ndarray               # (N,) int64
    y_days: np.ndarray                 # (N,) float32 — FIX 6: target de regresión
    y_multitask: np.ndarray            # (N, 4) float32
    lengths: np.ndarray                # (N,) int32
    y_true_tte: np.ndarray             # (N,) float64
    y_true_event_observed: np.ndarray  # (N,) int8
    meta_user: np.ndarray              # (N,) object
    meta_event_ts: np.ndarray          # (N,) object
    meta_attempt_seq: np.ndarray       # (N,) int64


def build_sequences(
    df: pl.DataFrame,
    feature_cols: List[str],
    lookback: int,
    config: dict,
    min_sequence_length: int = 3,
) -> SequenceOutput:
    """
    Construye secuencias con sliding windows — v1.5.0.

    Cambios vs v1.4.5:
      - Bucket encoding usa tte_to_bucket_adaptive() (FIX 2)
      - Incluye y_days para head de regresión (FIX 6)
      - Eliminado parámetro horizon_weeks (ya no aplica)
    """
    t_start = time.time()
    n_features = len(feature_cols)

    print(f"\n🔄 Construyendo secuencias (lookback={lookback})...")

    # Sort global
    df_sorted = df.sort(['userIdentifier', 'event_ts', 'attempt_seq'])

    # Extraer arrays
    X_all = df_sorted.select(feature_cols).fill_null(0).to_numpy().astype(np.float32)
    tte_all = df_sorted['time_to_event_days'].to_numpy().astype(np.float64)
    eo_all = df_sorted['event_observed'].to_numpy().astype(np.int8)

    y3d_all = df_sorted['y_3d'].to_numpy().astype(np.float32)
    y7d_all = df_sorted['y_7d'].to_numpy().astype(np.float32)
    y14d_all = df_sorted['y_14d'].to_numpy().astype(np.float32)
    y30d_all = df_sorted['y_30d'].to_numpy().astype(np.float32)

    users_all = df_sorted['userIdentifier'].to_numpy()
    events_all = df_sorted['event_ts'].to_list()
    attempt_seq_all = df_sorted['attempt_seq'].to_numpy().astype(np.int64)

    # Group offsets
    users_arr = df_sorted['userIdentifier'].to_numpy()
    change_points = np.where(users_arr[:-1] != users_arr[1:])[0] + 1
    starts = np.concatenate([[0], change_points])
    ends = np.concatenate([change_points, [len(users_arr)]])

    n_users_total = len(starts)
    total_seqs = df_sorted.height  # upper bound

    # Pre-allocate
    X_out = np.zeros((total_seqs, lookback, n_features), dtype=np.float32)
    y_bucket_out = np.zeros(total_seqs, dtype=np.int64)
    y_days_out = np.full(total_seqs, np.nan, dtype=np.float32)  # FIX 6
    y_mt_out = np.zeros((total_seqs, 4), dtype=np.float32)
    lengths_out = np.zeros(total_seqs, dtype=np.int32)
    tte_out = np.full(total_seqs, np.nan, dtype=np.float64)
    eo_out = np.zeros(total_seqs, dtype=np.int8)
    meta_user_out = np.empty(total_seqs, dtype=object)
    meta_event_out = np.empty(total_seqs, dtype=object)
    meta_seq_out = np.zeros(total_seqs, dtype=np.int64)

    write_idx = 0
    skipped_users = 0
    t_group = time.time()

    for u_idx in range(n_users_total):
        s, e = starts[u_idx], ends[u_idx]
        n_evt = e - s

        if n_evt < min_sequence_length:
            skipped_users += 1
            continue

        X_user = X_all[s:e]
        tte_user = tte_all[s:e]
        eo_user = eo_all[s:e]
        w_end = write_idx + n_evt

        # Sliding window
        for i in range(n_evt):
            seq_start = max(0, i - lookback + 1)
            seq_len = i - seq_start + 1
            X_out[write_idx + i, lookback - seq_len:, :] = X_user[seq_start:i+1]

        # FIX 2: Bucket encoding adaptativo
        buckets = tte_to_bucket_adaptive(tte_user, config)
        y_bucket_out[write_idx:w_end] = buckets

        # FIX 6: Target de regresión directa (días)
        # Para eventos censurados, usamos baseline_interval_days como target
        y_days = np.where(
            np.isnan(tte_user),
            config['baseline_interval_days'],
            np.clip(tte_user, 0, 60)  # cap a 60d para estabilidad
        ).astype(np.float32)
        y_days_out[write_idx:w_end] = y_days

        # Multitask
        y_mt_out[write_idx:w_end, 0] = y3d_all[s:e]
        y_mt_out[write_idx:w_end, 1] = y7d_all[s:e]
        y_mt_out[write_idx:w_end, 2] = y14d_all[s:e]
        y_mt_out[write_idx:w_end, 3] = y30d_all[s:e]

        lengths_out[write_idx:w_end] = np.minimum(
            np.arange(1, n_evt + 1), lookback
        ).astype(np.int32)

        tte_out[write_idx:w_end] = tte_user
        eo_out[write_idx:w_end] = eo_user
        meta_user_out[write_idx:w_end] = users_all[s:e]
        meta_event_out[write_idx:w_end] = events_all[s:e]
        meta_seq_out[write_idx:w_end] = attempt_seq_all[s:e]

        write_idx = w_end

        if (u_idx + 1) % 10_000 == 0:
            elapsed = time.time() - t_group
            pct = (u_idx + 1) / n_users_total * 100
            print(f"   [{pct:5.1f}%] {u_idx+1:,}/{n_users_total:,} users "
                  f"({write_idx:,} seqs) — {elapsed:.1f}s")

    # Trim
    if write_idx < total_seqs:
        X_out = X_out[:write_idx]
        y_bucket_out = y_bucket_out[:write_idx]
        y_days_out = y_days_out[:write_idx]
        y_mt_out = y_mt_out[:write_idx]
        lengths_out = lengths_out[:write_idx]
        tte_out = tte_out[:write_idx]
        eo_out = eo_out[:write_idx]
        meta_user_out = meta_user_out[:write_idx]
        meta_event_out = meta_event_out[:write_idx]
        meta_seq_out = meta_seq_out[:write_idx]

    result = SequenceOutput(
        X=X_out, y_bucket=y_bucket_out, y_days=y_days_out,
        y_multitask=y_mt_out, lengths=lengths_out,
        y_true_tte=tte_out, y_true_event_observed=eo_out,
        meta_user=meta_user_out, meta_event_ts=meta_event_out,
        meta_attempt_seq=meta_seq_out,
    )

    print(f"\n✅ Secuencias: {write_idx:,} ejemplos, shape {result.X.shape}")
    print(f"   Usuarios descartados (< {min_sequence_length} events): {skipped_users:,}")
    print_bucket_distribution(result.y_bucket, config)

    return result


# ═══ BUILD SEQUENCES ═══
feature_cols_norm = [f + '_norm' for f in HMM_FEATURES]

seq_train = build_sequences(df_train, feature_cols_norm, CONFIG['lookback_window'], CONFIG)
seq_val = build_sequences(df_val, feature_cols_norm, CONFIG['lookback_window'], CONFIG)
seq_test = build_sequences(df_test, feature_cols_norm, CONFIG['lookback_window'], CONFIG)

X_train, y_bucket_train = seq_train.X, seq_train.y_bucket
y_days_train = seq_train.y_days
y_multitask_train, lengths_train = seq_train.y_multitask, seq_train.lengths

X_val, y_bucket_val = seq_val.X, seq_val.y_bucket
y_days_val = seq_val.y_days
y_multitask_val, lengths_val = seq_val.y_multitask, seq_val.lengths

X_test, y_bucket_test = seq_test.X, seq_test.y_bucket
y_days_test = seq_test.y_days
y_multitask_test, lengths_test = seq_test.y_multitask, seq_test.lengths



🔄 Construyendo secuencias (lookback=8)...
   [  4.5%] 20,000/443,872 users (177,729 seqs) — 1.2s
   [  6.8%] 30,000/443,872 users (265,572 seqs) — 1.8s
   [  9.0%] 40,000/443,872 users (352,924 seqs) — 2.3s
   [ 15.8%] 70,000/443,872 users (619,286 seqs) — 4.1s
   [ 18.0%] 80,000/443,872 users (707,743 seqs) — 4.7s
   [ 20.3%] 90,000/443,872 users (797,370 seqs) — 5.3s
   [ 27.0%] 120,000/443,872 users (1,066,086 seqs) — 7.0s
   [ 29.3%] 130,000/443,872 users (1,153,752 seqs) — 7.6s
   [ 36.0%] 160,000/443,872 users (1,417,616 seqs) — 9.3s
   [ 40.6%] 180,000/443,872 users (1,596,108 seqs) — 10.4s
   [ 45.1%] 200,000/443,872 users (1,772,257 seqs) — 11.6s
   [ 47.3%] 210,000/443,872 users (1,859,891 seqs) — 12.2s
   [ 49.6%] 220,000/443,872 users (1,947,166 seqs) — 12.8s
   [ 51.8%] 230,000/443,872 users (2,036,091 seqs) — 13.4s
   [ 54.1%] 240,000/443,872 users (2,124,944 seqs) — 14.0s
   [ 56.3%] 250,000/443,872 users (2,214,787 seqs) — 14.6s
   [ 63.1%] 280,000/443,872 users (2,480

## 10. Dataset + DataLoaders

In [14]:
class RecurrentChargeDataset(Dataset):
    """Dataset for recurrent charge sequences — v1.5.0 con y_days."""

    def __init__(self, X, y_bucket, y_days, y_multitask=None, lengths=None):
        self.X = torch.from_numpy(X.copy()).float()
        self.y_bucket = torch.from_numpy(y_bucket.copy()).long()
        self.y_days = torch.from_numpy(y_days.copy()).float()  # FIX 6

        if y_multitask is not None:
            self.y_multitask = torch.from_numpy(y_multitask.copy()).float()
        else:
            self.y_multitask = None

        if lengths is not None:
            self.lengths = torch.from_numpy(lengths.copy()).long()
        else:
            self.lengths = None

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        item = {
            'X': self.X[idx],
            'y_bucket': self.y_bucket[idx],
            'y_days': self.y_days[idx],
        }
        if self.y_multitask is not None:
            item['y_multitask'] = self.y_multitask[idx]
        if self.lengths is not None:
            item['length'] = self.lengths[idx]
        return item


# ═══ CREATE DATALOADERS ═══
print("\n🔧 Creando DataLoaders...")

train_dataset = RecurrentChargeDataset(X_train, y_bucket_train, y_days_train, y_multitask_train, lengths_train)
val_dataset = RecurrentChargeDataset(X_val, y_bucket_val, y_days_val, y_multitask_val, lengths_val)
test_dataset = RecurrentChargeDataset(X_test, y_bucket_test, y_days_test, y_multitask_test, lengths_test)

USE_CUDA = torch.cuda.is_available()
LOADER_KWARGS = {
    'num_workers': 4 if USE_CUDA else 0,
    'pin_memory': True if USE_CUDA else False,
    'persistent_workers': True if USE_CUDA else False,
}
TRAIN_BATCH_SIZE = 2048 if USE_CUDA else CONFIG['batch_size']

train_loader = DataLoader(train_dataset, batch_size=TRAIN_BATCH_SIZE, shuffle=True, **LOADER_KWARGS)
val_loader = DataLoader(val_dataset, batch_size=TRAIN_BATCH_SIZE * 2, shuffle=False, **LOADER_KWARGS)

print(f"✅ DataLoaders: Train {len(train_dataset):,}, Val {len(val_dataset):,}, Test {len(test_dataset):,}")



🔧 Creando DataLoaders...
✅ DataLoaders: Train 3,939,761, Val 828,438, Test 920,799


## 11. Arquitecturas (5 modelos x dual head)

GRU, RNN, LSTM, TCN, DEEP_GRU. Cada uno con cabeza de bucket + regresion.

In [15]:
# ARQUITECTURAS 4 modelos × dual head (bucket + regresión)
class TemporalAttention(nn.Module):
    """Atención aprendida sobre T timesteps — sin cambios."""
    def __init__(self, hidden_size: int):
        super().__init__()
        self.score = nn.Linear(hidden_size, 1, bias=False)

    def forward(self, out: torch.Tensor, lengths: torch.Tensor = None):
        scores = self.score(out).squeeze(-1)
        if lengths is not None:
            T = out.size(1)
            mask = torch.arange(T, device=out.device).unsqueeze(0) < lengths.unsqueeze(1)
            scores = scores.masked_fill(~mask, float('-inf'))
        weights = torch.softmax(scores, dim=1).unsqueeze(-1)
        return (out * weights).sum(dim=1)


class MultitaskHeadsMixin:
    """Mixin: cabezas auxiliares multitask + regression head."""

    def _build_multitask_heads(self, hidden_size: int):
        self.fc_y3d  = nn.Linear(hidden_size, 1)
        self.fc_y7d  = nn.Linear(hidden_size, 1)
        self.fc_y14d = nn.Linear(hidden_size, 1)
        self.fc_y30d = nn.Linear(hidden_size, 1)

    def _multitask_forward(self, pooled: torch.Tensor) -> torch.Tensor:
        return torch.cat([
            self.fc_y3d(pooled), self.fc_y7d(pooled),
            self.fc_y14d(pooled), self.fc_y30d(pooled),
        ], dim=1)

    def _shared_head_forward(self, pooled):
        """Shared post-processing: norm → dropout → bucket + days + multitask."""
        pooled = self.norm(pooled)
        pooled = self.dropout(pooled)
        logits_bucket = self.fc_bucket(pooled)
        pred_days = F.relu(self.fc_days(pooled)).squeeze(-1)
        logits_mt = self._multitask_forward(pooled) if self.use_multitask else None
        return logits_bucket, logits_mt, pred_days


# ─────────────────────────────────────────────────────────────────────────────
# MODEL 1: GRU — Gated Recurrent Unit
# ─────────────────────────────────────────────────────────────────────────────

class GRUModel(MultitaskHeadsMixin, nn.Module):
    """GRU + TemporalAttention + dual head (bucket + regression)."""

    def __init__(self, input_size, hidden_size, num_layers,
                 n_buckets, dropout=0.3, use_multitask=True):
        super().__init__()
        self.gru = nn.GRU(
            input_size, hidden_size, num_layers,
            batch_first=True,
            dropout=dropout if num_layers > 1 else 0.0,
        )
        self.attn       = TemporalAttention(hidden_size)
        self.norm       = nn.LayerNorm(hidden_size)
        self.dropout    = nn.Dropout(dropout)
        self.fc_bucket  = nn.Linear(hidden_size, n_buckets)
        self.fc_days    = nn.Linear(hidden_size, 1)  # FIX 6
        self.use_multitask = use_multitask
        if use_multitask:
            self._build_multitask_heads(hidden_size)

    def forward(self, x, lengths=None):
        out, _ = self.gru(x)
        pooled = self.attn(out, lengths)
        return self._shared_head_forward(pooled)


# ─────────────────────────────────────────────────────────────────────────────
# MODEL 2: RNN — Vanilla Recurrent Neural Network
# ─────────────────────────────────────────────────────────────────────────────

class RNNModel(MultitaskHeadsMixin, nn.Module):
    """
    Vanilla RNN + TemporalAttention + dual head.

    Sirve como ablación del GRU: si RNN ≈ GRU en MAE, los gates del GRU
    no aportan valor para secuencias cortas (lookback=8).
    """

    def __init__(self, input_size, hidden_size, num_layers,
                 n_buckets, dropout=0.3, use_multitask=True):
        super().__init__()
        self.rnn = nn.RNN(
            input_size, hidden_size, num_layers,
            batch_first=True, nonlinearity='tanh',
            dropout=dropout if num_layers > 1 else 0.0,
        )
        self.attn       = TemporalAttention(hidden_size)
        self.norm       = nn.LayerNorm(hidden_size)
        self.dropout    = nn.Dropout(dropout)
        self.fc_bucket  = nn.Linear(hidden_size, n_buckets)
        self.fc_days    = nn.Linear(hidden_size, 1)  # FIX 6
        self.use_multitask = use_multitask
        if use_multitask:
            self._build_multitask_heads(hidden_size)

    def forward(self, x, lengths=None):
        out, _ = self.rnn(x)
        pooled = self.attn(out, lengths)
        return self._shared_head_forward(pooled)


# ─────────────────────────────────────────────────────────────────────────────
# MODEL 3: LSTM — Long Short-Term Memory
# ─────────────────────────────────────────────────────────────────────────────

class LSTMModel(MultitaskHeadsMixin, nn.Module):
    """LSTM + TemporalAttention + dual head."""

    def __init__(self, input_size, hidden_size, num_layers,
                 n_buckets, dropout=0.3, use_multitask=True):
        super().__init__()
        self.lstm = nn.LSTM(
            input_size, hidden_size, num_layers,
            batch_first=True,
            dropout=dropout if num_layers > 1 else 0.0,
        )
        self.attn       = TemporalAttention(hidden_size)
        self.norm       = nn.LayerNorm(hidden_size)
        self.dropout    = nn.Dropout(dropout)
        self.fc_bucket  = nn.Linear(hidden_size, n_buckets)
        self.fc_days    = nn.Linear(hidden_size, 1)  # FIX 6
        self.use_multitask = use_multitask
        if use_multitask:
            self._build_multitask_heads(hidden_size)

    def forward(self, x, lengths=None):
        out, _ = self.lstm(x)
        pooled = self.attn(out, lengths)
        return self._shared_head_forward(pooled)


# ─────────────────────────────────────────────────────────────────────────────
# MODEL 4: TCN — Temporal Convolutional Network
# ─────────────────────────────────────────────────────────────────────────────

class CausalConvBlock(nn.Module):
    """Bloque causal dilatado con residual. Padding solo a la izquierda."""

    def __init__(self, in_channels, out_channels, kernel_size, dilation, dropout=0.3):
        super().__init__()
        self.causal_pad = (kernel_size - 1) * dilation
        self.conv = nn.Conv1d(
            in_channels, out_channels,
            kernel_size=kernel_size, dilation=dilation,
            padding=0, bias=True,
        )
        self.bn      = nn.BatchNorm1d(out_channels)
        self.relu    = nn.ReLU()
        self.dropout = nn.Dropout(dropout)
        self.res_proj = (
            nn.Conv1d(in_channels, out_channels, kernel_size=1, bias=False)
            if in_channels != out_channels else nn.Identity()
        )

    def forward(self, x):
        x_padded = F.pad(x, (self.causal_pad, 0))
        out = self.conv(x_padded)
        out = self.bn(out)
        out = self.relu(out)
        out = self.dropout(out)
        return out + self.res_proj(x)


class TCNModel(MultitaskHeadsMixin, nn.Module):
    """
    Temporal Convolutional Network + dual head.

    Con dilation=[1,2,4] y kernel=3: RF = 1 + (3-1)×(1+2+4) = 15 > lookback=8.
    Paralelizable, sin vanishing gradient, campo receptivo explícito.

    Ref: Bai et al. (2018) arXiv:1803.01271
    """

    def __init__(self, input_size, num_channels, kernel_size,
                 n_buckets, dropout=0.3, use_multitask=True):
        super().__init__()
        layers = []
        in_ch = input_size
        for i, out_ch in enumerate(num_channels):
            dilation = 2 ** i
            layers.append(CausalConvBlock(in_ch, out_ch, kernel_size, dilation, dropout))
            in_ch = out_ch
        self.tcn = nn.Sequential(*layers)
        hidden_size = num_channels[-1]

        self.attn       = TemporalAttention(hidden_size)
        self.norm       = nn.LayerNorm(hidden_size)
        self.dropout    = nn.Dropout(dropout)
        self.fc_bucket  = nn.Linear(hidden_size, n_buckets)
        self.fc_days    = nn.Linear(hidden_size, 1)  # FIX 6
        self.use_multitask = use_multitask
        if use_multitask:
            self._build_multitask_heads(hidden_size)

    def forward(self, x, lengths=None):
        x_t = x.permute(0, 2, 1)       # (B, C, T)
        out_t = self.tcn(x_t)           # (B, H, T)
        out = out_t.permute(0, 2, 1)    # (B, T, H)
        pooled = self.attn(out, lengths)
        return self._shared_head_forward(pooled)


# ─────────────────────────────────────────────────────────────────────────────

# ─────────────────────────────────────────────────────────────────────────────
# MODEL 5: DEEP_GRU — Residual GRU profundo (Task 5 v1.7.4)
# Pre-LN (Xiong 2020) + Residual + DropPath (Huang 2016) + TemporalAttention
# ─────────────────────────────────────────────────────────────────────────────
class DropPath(nn.Module):
    """Stochastic Depth. Drops entire residual branches during training."""
    def __init__(self, p=0.1):
        super().__init__()
        self.p = p
    def forward(self, x):
        if not self.training or self.p == 0:
            return x
        keep = 1 - self.p
        shape = (x.shape[0],) + (1,) * (x.ndim - 1)
        mask = torch.bernoulli(torch.full(shape, keep, device=x.device))
        return x * mask / keep

class ResidualGRUBlock(nn.Module):
    """Pre-LN → GRU → Dropout → DropPath → residual add."""
    def __init__(self, hidden, dropout=0.3, drop_path=0.1):
        super().__init__()
        self.ln = nn.LayerNorm(hidden)
        self.gru = nn.GRU(hidden, hidden, batch_first=True)
        self.dp = nn.Dropout(dropout)
        self.drop_path = DropPath(drop_path)
    def forward(self, x):
        h, _ = self.gru(self.ln(x))
        return x + self.drop_path(self.dp(h))

class DeepResidualGRU(MultitaskHeadsMixin, nn.Module):
    """GRU profundo con Pre-LN + residual + DropPath + TemporalAttention + dual head."""
    def __init__(self, input_size, hidden=128, n_layers=4,
                 n_buckets=10, dropout=0.3, drop_path=0.1, use_multitask=True):
        super().__init__()
        self.input_proj = nn.Linear(input_size, hidden)
        self.blocks = nn.ModuleList([
            ResidualGRUBlock(hidden, dropout, drop_path * (i+1) / n_layers)
            for i in range(n_layers)
        ])
        self.attn       = TemporalAttention(hidden)
        self.norm       = nn.LayerNorm(hidden)
        self.dropout    = nn.Dropout(dropout)
        self.fc_bucket  = nn.Linear(hidden, n_buckets)
        self.fc_days    = nn.Linear(hidden, 1)
        self.use_multitask = use_multitask
        if use_multitask:
            self._build_multitask_heads(hidden)
    def forward(self, x, lengths=None):
        h = self.input_proj(x)
        for blk in self.blocks:
            h = blk(h)
        pooled = self.attn(h, lengths)
        return self._shared_head_forward(pooled)


# FACTORY + AUDIT
# ─────────────────────────────────────────────────────────────────────────────

def create_model(model_type: str, input_size: int, config: dict) -> nn.Module:
    """Factory unificada. mc se lee dentro de cada branch para tolerar
    model_types con configs de shape distinta (v1.7.4)."""
    n_buckets     = config['n_time_buckets'] + 1
    use_multitask = config['use_multitask']

    if model_type in ('GRU', 'RNN', 'LSTM'):
        mc = config['model_configs'][model_type]
        cls = {'GRU': GRUModel, 'RNN': RNNModel, 'LSTM': LSTMModel}[model_type]
        return cls(
            input_size, mc['hidden_size'], mc['num_layers'],
            n_buckets, mc['dropout'], use_multitask,
        )

    if model_type == 'TCN':
        mc = config['model_configs']['TCN']
        return TCNModel(
            input_size, mc['num_channels'], mc['kernel_size'],
            n_buckets, mc['dropout'], use_multitask,
        )

    if model_type == 'DEEP_GRU':
        mc = config['model_configs']['DEEP_GRU']
        return DeepResidualGRU(
            input_size=input_size,
            hidden=mc['hidden_size'],
            n_layers=mc['num_layers'],
            n_buckets=n_buckets,
            dropout=mc['dropout'],
            drop_path=mc['drop_path'],
            use_multitask=use_multitask,
        )

    raise ValueError(
        f"Model '{model_type}' no implementado. "
        f"Opciones: GRU, RNN, LSTM, TCN, DEEP_GRU"
    )


# AUDIT
print("✅ Arquitecturas v1.5.1 cargadas (4 modelos × dual head)\n")
print(f"  {'Modelo':<8} {'Params':>10}  {'Detalle'}")
print(f"  {'─'*60}")

for mt in CONFIG['models_to_train']:
    m = create_model(mt, len(HMM_FEATURES), CONFIG)
    n = sum(p.numel() for p in m.parameters() if p.requires_grad)
    has_fc_days = hasattr(m, 'fc_days')
    if mt == 'TCN':
        mc = CONFIG['model_configs']['TCN']
        rf = 1 + sum((mc['kernel_size'] - 1) * (2**i) for i in range(len(mc['num_channels'])))
        detail = f"ch={mc['num_channels']}, RF={rf}, fc_days={'✅' if has_fc_days else '❌'}"
    else:
        mc = CONFIG['model_configs'][mt]
        detail = f"h={mc['hidden_size']}, L={mc['num_layers']}, d={mc['dropout']}, fc_days={'✅' if has_fc_days else '❌'}"
    print(f"  {mt:<8} {n:>10,}  {detail}")

print(f"\n  Forward: (x, lengths) → (logits_bucket, logits_mt, pred_days)")
print("DeepResidualGRU + DropPath vive ahora en celda 28 junto a los otros modelos.")


✅ Arquitecturas v1.5.1 cargadas (4 modelos × dual head)

  Modelo       Params  Detalle
  ────────────────────────────────────────────────────────────
  GRU       1,423,119  h=256, L=4, d=0.4, fc_days=✅
  RNN         477,455  h=256, L=4, d=0.4, fc_days=✅
  LSTM      1,895,951  h=256, L=4, d=0.4, fc_days=✅
  TCN         387,727  ch=[64, 128, 256, 128], RF=46, fc_days=✅
  DEEP_GRU  1,597,967  h=256, L=4, d=0.4, fc_days=✅

  Forward: (x, lengths) → (logits_bucket, logits_mt, pred_days)
DeepResidualGRU + DropPath vive ahora en celda 28 junto a los otros modelos.


## 12. Loss Functions

**Fix P0-1.** Antes habia dos `compute_loss_v2` con cuerpos distintos en celdas separadas. Aqui hay **una sola** `compute_loss` (el cuerpo que el training realmente usaba: focal + label smoothing ordinal + class weights + ordinal asimetrica + multitask + Huber). La `compute_loss` vieja sin uso fue eliminada.

Ref: Lin et al. (2017) Focal Loss; Szegedy et al. (2016) Label Smoothing.

In [16]:
# Loss del modelo de produccion. ordinal_loss_asymmetric es dependencia de
# compute_loss, por eso se define primero.

def ordinal_loss_asymmetric(
    logits: torch.Tensor,
    targets: torch.Tensor,
    n_buckets: int,
    late_alpha: float = 2.0,
) -> torch.Tensor:
    """
    FIX 3: Ordinal loss con penalización asimétrica.

    Antes (v1.4.5): penalizaba simétricamente |bucket_pred - bucket_real|
    Ahora: penaliza ×alpha cuando bucket_pred > bucket_real (late = inútil)
           penaliza ×1.0  cuando bucket_pred ≤ bucket_real (early = útil)

    Intuición para reminders:
        Si predices bucket 7 (10.5d) pero el real es bucket 2 (1.5d),
        el reminder llegaría 9 días tarde → inútil. Esto debe penalizarse
        más que predecir bucket 1 (0.5d) cuando el real era bucket 3 (2.5d),
        porque el reminder llegaría 2 días antes → potencialmente útil.

    Args:
        logits: (B, K) raw logits del clasificador
        targets: (B,) int — bucket index real
        n_buckets: K
        late_alpha: penalización para predicciones tardías (default 2.0)

    Returns:
        loss: scalar
    """
    probs = torch.softmax(logits, dim=1)  # (B, K)

    bucket_idx = torch.arange(
        n_buckets, device=logits.device, dtype=torch.float32
    )  # (K,)

    # Distancias signadas: positivo = pred > real (tardío)
    signed_dist = bucket_idx.unsqueeze(0) - targets.float().unsqueeze(1)  # (B, K)

    # Pesos asimétricos: late (positivo) penaliza más
    weights = torch.where(signed_dist > 0, late_alpha, 1.0)

    # Distancia ponderada
    weighted_dist = weights * torch.abs(signed_dist)  # (B, K)

    # Expected weighted distance bajo distribución predicha
    return (probs * weighted_dist).sum(dim=1).mean()


def compute_loss(
    logits_bucket, logits_multitask, pred_days,
    targets_bucket, targets_multitask, targets_days,
    config,
    class_weights=None,   # nuevo: tensor (n_buckets,) desde el training loop
):
    """
    Loss compuesta v2.1 (Ronda 1 accuracy push):
      - bucket_ce    : CrossEntropy o Focal, con class_weights y label_smoothing
      - multitask_ce : igual al anterior (sin cambios)
      - regression   : Huber sobre pred_days
      - ordinal_asym : late penalty (sin cambios)
    Ref: Lin et al. 2017 (Focal Loss), Szegedy et al. 2016 (Label Smoothing).
    """
    n_buckets = logits_bucket.shape[1]
    ls = config.get('label_smoothing', 0.0)
    use_focal = config.get('use_focal_loss', False)
    gamma = config.get('focal_gamma', 2.0)

    # --- Bucket CE con focal + weights + label smoothing ordinal ---
    log_probs = torch.nn.functional.log_softmax(logits_bucket, dim=-1)

    if ls > 0:
        # Ordinal-aware smoothing: masa en bucket real + vecinos adyacentes
        with torch.no_grad():
            smooth = torch.full_like(log_probs, ls / (n_buckets - 1))
            smooth.scatter_(1, targets_bucket.unsqueeze(1), 1.0 - ls)
        nll = -(smooth * log_probs).sum(dim=-1)
    else:
        nll = torch.nn.functional.nll_loss(log_probs, targets_bucket, reduction='none')

    if use_focal:
        # Focal Loss: (1-p)^gamma * CE
        with torch.no_grad():
            probs = torch.softmax(logits_bucket, dim=-1)
            p_true = probs.gather(1, targets_bucket.unsqueeze(1)).squeeze(1)
            focal_w = (1 - p_true).pow(gamma)
        nll = focal_w * nll

    if class_weights is not None:
        w = class_weights[targets_bucket]
        nll = w * nll

    bucket_loss = nll.mean()

    # --- Multitask CE (sin cambios funcionales) ---
    multi_loss = torch.nn.functional.cross_entropy(logits_multitask, targets_multitask)

    # --- Regression Huber (sin cambios) ---
    valid_reg = ~torch.isnan(targets_days)
    if valid_reg.any():
        reg_loss = torch.nn.functional.huber_loss(
            pred_days[valid_reg], targets_days[valid_reg], delta=3.0
        )
    else:
        reg_loss = torch.tensor(0.0, device=logits_bucket.device)

    # --- Ordinal asymmetric late penalty (sin cambios) ---
    # --- Ordinal asymmetric late penalty (sin cambios) ---
    ord_loss = ordinal_loss_asymmetric(
        logits_bucket,
        targets_bucket,
        n_buckets=n_buckets,
        late_alpha=config.get('late_penalty_alpha', 2.0),
    )

    total = (
        bucket_loss
        + config.get('multitask_weight', 0.3) * multi_loss
        + config.get('regression_weight', 0.5) * reg_loss
        + config.get('ordinal_loss_weight', 0.5) * ord_loss
    )
    return total

## 13. Training Infrastructure

`EarlyStopping`, `train_epoch`, `evaluate`, `train_model`. Las llamadas a `compute_loss_v2` se actualizaron al nombre unico `compute_loss`.

In [17]:
class EarlyStopping:
    def __init__(self, patience: int = 10, min_delta: float = 1e-4):
        self.patience = patience
        self.min_delta = min_delta
        self.counter = 0
        self.best_score = None
        self.early_stop = False
        self.best_epoch = 0

    def __call__(self, score: float, epoch: int) -> bool:
        if self.best_score is None:
            self.best_score = score
            self.best_epoch = epoch
            return False
        if score < (self.best_score - self.min_delta):
            self.best_score = score
            self.best_epoch = epoch
            self.counter = 0
        else:
            self.counter += 1
            if self.counter >= self.patience:
                self.early_stop = True
        return self.early_stop


def train_epoch(model, train_loader, optimizer, config, device, scaler, class_weights_t=None):
    """Train one epoch — v1.7.4 (clean AMP + Ronda 1)."""
    model.train()
    total_loss = 0
    use_amp = scaler is not None
    
    for batch in train_loader:
        X = batch['X'].to(device, non_blocking=True)
        y_bucket = batch['y_bucket'].to(device, non_blocking=True)
        y_days = batch['y_days'].to(device, non_blocking=True)
        y_multitask = batch['y_multitask'].to(device, non_blocking=True) \
            if 'y_multitask' in batch else None
        
        optimizer.zero_grad(set_to_none=True)
        
        if use_amp:
            with torch.amp.autocast('cuda'):
                logits_bucket, logits_multitask, pred_days = model(X)
                loss = compute_loss(
                    logits_bucket, logits_multitask, pred_days,
                    y_bucket, y_multitask, y_days, config,
                    class_weights=class_weights_t,
                )
            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            if config.get('grad_clip_max_norm', 0) > 0:
                torch.nn.utils.clip_grad_norm_(
                    model.parameters(),
                    max_norm=config['grad_clip_max_norm'],
                )
            scaler.step(optimizer)
            scaler.update()
        else:
            logits_bucket, logits_multitask, pred_days = model(X)
            loss = compute_loss(
                logits_bucket, logits_multitask, pred_days,
                y_bucket, y_multitask, y_days, config,
                class_weights=class_weights_t,
            )
            loss.backward()
            if config.get('grad_clip_max_norm', 0) > 0:
                torch.nn.utils.clip_grad_norm_(
                    model.parameters(),
                    max_norm=config['grad_clip_max_norm'],
                )
            optimizer.step()
        
        total_loss += loss.item()
    
    return total_loss / len(train_loader)


@torch.no_grad()
def evaluate(model, val_loader, config, device, use_amp, class_weights_t=None):
    """Evaluate — v1.7.0 con focal loss."""
    model.eval()
    total_loss = 0
    correct = 0
    total = 0

    for batch in val_loader:
        X = batch['X'].to(device, non_blocking=True)
        y_bucket = batch['y_bucket'].to(device, non_blocking=True)
        y_days = batch['y_days'].to(device, non_blocking=True)
        y_multitask = batch['y_multitask'].to(device, non_blocking=True) \
            if 'y_multitask' in batch else None

        if use_amp:
            with torch.amp.autocast('cuda'):
                logits_bucket, logits_multitask, pred_days = model(X)
                loss = compute_loss(
                    logits_bucket, logits_multitask, pred_days,
                    y_bucket, y_multitask, y_days, config,
                    class_weights=class_weights_t,   
                )
        else:
            logits_bucket, logits_multitask, pred_days = model(X)
            loss = compute_loss(
                logits_bucket, logits_multitask, pred_days,
                y_bucket, y_multitask, y_days, config,
                class_weights=class_weights_t,   
            )

        total_loss += loss.item()
        preds = logits_bucket.argmax(dim=1)
        correct += (preds == y_bucket).sum().item()
        total += y_bucket.size(0)

    return total_loss / len(val_loader), correct / total


def train_model(model, train_loader, val_loader, config, device):
    """Entrena modelo — v1.5.0."""
    model = model.to(device)
    use_amp = torch.cuda.is_available() and device != 'cpu'
    amp_scaler = torch.amp.GradScaler('cuda') if use_amp else None

    optimizer = optim.Adam(
        model.parameters(),
        lr=config['learning_rate'],
        weight_decay=config['weight_decay'],
    )
# Class weights inversos (sqrt) para bucket head
    class_weights_t = None
    if config.get('use_class_weights', False):
        # Contar frecuencia de cada bucket en train
        all_targets = []
        for batch in train_loader:
            all_targets.append(batch['y_bucket'])
            if len(all_targets) >= 20: break  # sample; suficiente para estimación
        all_targets = torch.cat(all_targets).cpu().numpy()
        n_buckets = config['n_time_buckets'] + 1
        counts = np.bincount(all_targets, minlength=n_buckets).astype(np.float64)
        counts = np.maximum(counts, 1.0)  # evita div/0
        power = config.get('class_weight_power', 0.5)
        inv_freq = (counts.sum() / (n_buckets * counts)) ** power
        # Normalizar a media 1.0 para no cambiar la escala del loss
        inv_freq = inv_freq / inv_freq.mean()
        class_weights_t = torch.from_numpy(inv_freq).float().to(device)
        print(f"    class_weights (power={power}): {inv_freq.round(2).tolist()}")
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode='min', patience=5, factor=0.3, min_lr=1e-6,
    )

    early_stopping = EarlyStopping(patience=15)
    best_model_state = None
    best_val_loss = float('inf')
    prev_lr = config['learning_rate']

    mode_str = "AMP FP16" if use_amp else "FP32"
    print(f"\n🚀 Entrenando ({mode_str}, batch={train_loader.batch_size})...")
    t_start = time.time()

    for epoch in range(config['max_epochs']):
        t_epoch = time.time()

        train_loss = train_epoch(model, train_loader, optimizer, config, device, amp_scaler, class_weights_t)
        val_loss, val_acc = evaluate(model, val_loader, config, device, use_amp, class_weights_t)
        scheduler.step(val_loss)
        current_lr = optimizer.param_groups[0]['lr']

        lr_changed = abs(current_lr - prev_lr) > 1e-10
        if (epoch + 1) % 10 == 0 or epoch == 0 or lr_changed:
            elapsed = time.time() - t_start
            epoch_time = time.time() - t_epoch
            lr_tag = f" ⬇️ LR→{current_lr:.1e}" if lr_changed else ""
            print(f"  Epoch {epoch+1:>3}/{config['max_epochs']}: "
                  f"train={train_loss:.4f}, val={val_loss:.4f}, acc={val_acc:.3f} "
                  f"(lr={current_lr:.1e}, {epoch_time:.1f}s/ep){lr_tag}")
        prev_lr = current_lr

        if val_loss < best_val_loss:
            best_val_loss = val_loss
            best_model_state = {k: v.clone() for k, v in model.state_dict().items()}

        if early_stopping(val_loss, epoch):
            print(f"  ⏹️  Early stopping en epoch {epoch + 1}")
            break

    if best_model_state is not None:
        model.load_state_dict(best_model_state)

    total_time = time.time() - t_start
    print(f"✅ Training completo. Best val loss: {best_val_loss:.4f} ({total_time/60:.1f} min)")
    return model, best_val_loss

## 14. Entrenar Modelos

In [18]:
# ENTRENAR 4 MODELOS × 1 SEED CADA UNO
# FIX v1.5.6: el loop original decía '3 seeds' pero solo entrena 1 seed/modelo

device = 'cuda' if torch.cuda.is_available() else 'cpu'
input_size = len(HMM_FEATURES)
best_models = {}
training_log = {}

total_start = time.time()

for model_type in CONFIG['models_to_train']:
    print(f"\n{'='*80}")
    print(f"  ENTRENANDO {model_type} — v1.5.1 (FIX 5 + FIX 6)")
    print(f"{'='*80}")

    model = create_model(model_type, input_size, CONFIG)
    n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f"  Parámetros: {n_params:,}")

    t0 = time.time()
    trained_model, val_loss = train_model(model, train_loader, val_loader, CONFIG, device)
    elapsed = (time.time() - t0) / 60

    best_models[model_type] = trained_model
    training_log[model_type] = {
        'params': n_params, 'val_loss': val_loss, 'time_min': elapsed,
    }

    torch.save({
        'model_state_dict': trained_model.state_dict(),
        'model_type': model_type,
        'val_loss': val_loss,
        'config': CONFIG,
    }, OUTPUT_DIR / f'{model_type}_model_v1_9_0.pt')

    print(f"  ✅ {model_type} guardado: val_loss={val_loss:.4f}, {elapsed:.1f} min")

    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

total_elapsed = (time.time() - total_start) / 60

print(f"\n\n{'='*80}")
print(f"  RESUMEN DE TRAINING ({total_elapsed:.1f} min total)")
print(f"{'='*80}")
print(f"\n  {'Modelo':<8} {'Params':>10} {'Val Loss':>10} {'Tiempo':>10}")
print(f"  {'─'*45}")
for mt, log in training_log.items():
    print(f"  {mt:<8} {log['params']:>10,} {log['val_loss']:>10.4f} {log['time_min']:>8.1f} min")

best_mt = min(training_log, key=lambda k: training_log[k]['val_loss'])
print(f"\n  🏆 Mejor por val_loss: {best_mt} ({training_log[best_mt]['val_loss']:.4f})")


  ENTRENANDO GRU — v1.5.1 (FIX 5 + FIX 6)
  Parámetros: 1,423,119
    class_weights (power=0.85): [1.22, 3.17, 1.54, 0.27, 0.43, 0.79, 1.5, 0.53, 0.35, 0.21]

🚀 Entrenando (AMP FP16, batch=2048)...
  Epoch   1/100: train=9.2414, val=8.8445, acc=0.484 (lr=1.0e-03, 67.4s/ep)
  Epoch  10/100: train=7.9727, val=8.8908, acc=0.497 (lr=1.0e-03, 52.8s/ep)
  Epoch  20/100: train=7.8396, val=8.2764, acc=0.525 (lr=1.0e-03, 52.0s/ep)
  Epoch  30/100: train=7.7754, val=7.7998, acc=0.538 (lr=1.0e-03, 54.0s/ep)
  Epoch  34/100: train=7.7560, val=7.8738, acc=0.530 (lr=3.0e-04, 50.3s/ep) ⬇️ LR→3.0e-04
  Epoch  40/100: train=7.6207, val=7.8676, acc=0.537 (lr=9.0e-05, 49.8s/ep) ⬇️ LR→9.0e-05
  ⏹️  Early stopping en epoch 43
✅ Training completo. Best val loss: 7.7328 (37.5 min)
  ✅ GRU guardado: val_loss=7.7328, 37.7 min

  ENTRENANDO RNN — v1.5.1 (FIX 5 + FIX 6)
  Parámetros: 477,455
    class_weights (power=0.85): [1.17, 3.1, 1.51, 0.28, 0.42, 0.83, 1.55, 0.57, 0.35, 0.22]

🚀 Entrenando (AMP FP16, batc

## 15. Predicciones + Decoders

`predict_full`, `decode_quantile`, `decode_expected_value`, `compute_full_metrics`. Predicciones provisionales; la Seccion 18 (16f) las sobrescribe con el decoder calibrado optimo.

In [19]:
# SECTION 15 — PREDICCIONES (SLIMMED v1.7.3)
# Audit: tablas 1/2/3 eliminadas (duplicaban 16f/16j/1b), "Recomendacion Final"
# eliminada (contenia bug P0-2 doble-llave y decidia sobre modelos no calibrados).
# Se conservan solo helpers + loop que alimenta a 16f.

@torch.no_grad()
def predict_full(model, X, device, batch_size=8192):
    model = model.to(device).eval()
    all_probs, all_days = [], []
    for s in range(0, len(X), batch_size):
        xb = torch.from_numpy(X[s:s+batch_size]).float().to(device)
        logits_b, _, pd_ = model(xb)
        all_probs.append(torch.softmax(logits_b, 1).cpu().numpy())
        all_days.append(pd_.cpu().numpy())
        del xb, logits_b
    return np.concatenate(all_probs, 0), np.concatenate(all_days, 0)

def decode_expected_value(probs, config):
    c = np.array(config['bucket_centers'], dtype=np.float64)
    return (probs * c).sum(axis=1)

def decode_quantile(probs, config, q=0.35):
    centers = np.array(config['bucket_centers'], dtype=np.float64)
    idx = np.argsort(centers); sc = centers[idx]; sp = probs[:, idx]
    cp = np.cumsum(sp, axis=1); N, K = probs.shape
    out = np.zeros(N)
    for i in range(N):
        j = min(np.searchsorted(cp[i], q, side='left'), K-1)
        if j == 0:
            out[i] = sc[0]
        else:
            lo, hi = cp[i, j-1], cp[i, j]
            cl, ch = sc[j-1], sc[j]
            out[i] = cl + (q-lo)/(hi-lo)*(ch-cl) if hi-lo > 1e-10 else sc[j]
    return out

bucket_probs_all = {}
reg_days_all = {}
for mt, model in best_models.items():
    bp, rd = predict_full(model, X_test, device)
    bucket_probs_all[mt] = bp
    reg_days_all[mt] = rd
    print(f"  {mt}: bp={bp.shape}, reg median={np.median(rd):.2f}d")

if torch.cuda.is_available(): torch.cuda.empty_cache()

actual_days = seq_test.y_true_tte
event_observed = seq_test.y_true_event_observed
mask = (event_observed == 1) & ~np.isnan(actual_days)
baseline_naive = np.full(len(X_test), CONFIG['baseline_interval_days'])
print(f"  Test: {len(actual_days):,} | obs: {mask.sum():,} | TTE med: {np.nanmedian(actual_days[mask]):.1f}d")
print("  Fuente de verdad = 16f (Ensemble Calibrado + Grid Search).")


# --- predictions provisionales (pre-calibración) ---
# 16a y downstream necesitan este dict. 16f lo sobreescribirá con el decoder
# calibrado óptimo. Aquí usamos q=0.35 como placeholder razonable.
predictions = {mt: decode_quantile(bucket_probs_all[mt], CONFIG, q=0.35)
               for mt in bucket_probs_all}

# compute_full_metrics helper (usado por 16a, 16f, 1b, etc.)
def compute_full_metrics(pred, actual, mask):
    p, a = pred[mask], actual[mask]
    err = np.abs(p - a)
    signed = p - a
    diff_biz = a - p
    late = signed > 0
    weights = np.where(signed > 0, 2.0, 1.0)
    return {
        'mae': float(err.mean()),
        'p50e': float(np.median(err)),
        'p90e': float(np.percentile(err, 90)),
        'late%': float(late.mean()),
        'ewr7': float(((diff_biz >= 1) & (diff_biz <= 7)).mean()),
        'ewr14': float(((diff_biz >= 1) & (diff_biz <= 14)).mean()),
        'cov3': float((err <= 3).mean()),
        'cov7': float((err <= 7).mean()),
        'asym': float((weights * err).mean()),
        'med_p': float(np.median(p)),
    }

print(f"  predictions: {list(predictions.keys())} (pre-cal, q=0.35)")

# --- Variables que downstream espera (16b usa rec_decoder_q, 16j/68/76 usan best_model_type) ---
# Provisional: el modelo con menor MAE pre-cal. 16f lo sobreescribirá.
_mae_pre = {mt: np.abs(predictions[mt][mask] - actual_days[mask]).mean() for mt in predictions}
best_model_type = min(_mae_pre, key=_mae_pre.get)
rec_decoder_q = 0.35  # default; 16f/16d harán grid search real
print(f"  best_model_type (pre-cal): {best_model_type} (MAE={_mae_pre[best_model_type]:.2f}d)")


  GRU: bp=(920799, 10), reg median=9.63d
  RNN: bp=(920799, 10), reg median=9.39d
  LSTM: bp=(920799, 10), reg median=9.70d
  TCN: bp=(920799, 10), reg median=9.69d
  DEEP_GRU: bp=(920799, 10), reg median=9.89d
  Test: 920,799 | obs: 739,773 | TTE med: 6.0d
  Fuente de verdad = 16f (Ensemble Calibrado + Grid Search).
  predictions: ['GRU', 'RNN', 'LSTM', 'TCN', 'DEEP_GRU'] (pre-cal, q=0.35)
  best_model_type (pre-cal): DEEP_GRU (MAE=6.07d)


## 16. Metricas de Negocio + Castigo (16a)

Libreria de metricas reusables: EWR, directional error, coverage, ECE, C-index, asymmetric MAE, pinball, business cost.

In [20]:
# MÉTRICAS DE NEGOCIO + MÉTRICAS DE CASTIGO

import numpy as np
import json
from pathlib import Path
@torch.no_grad()
def predict_buckets(model, X, device, batch_size=8192):
    """Wrapper: retorna solo bucket_probs."""
    bucket_probs, _ = predict_full(model, X, device, batch_size)
    return bucket_probs

# MÉTRICAS DE NEGOCIO

def early_warning_rate(pred_days, actual_days, mask, window=(1, 7)):
    """
    % de predicciones donde el reminder llegaría ANTES del cargo
    dentro de una ventana útil de negocio.

    Una predicción es útil si: window[0] <= actual - pred <= window[1]
    Interpretación: el reminder llegó entre 1 y 7 días antes del intento.

    window=(1,7):  segmento inmediato — el más valioso (acción T-1)
    window=(1,14): cobertura amplia — incluye segmento 8-14d
    """
    diff = actual_days[mask] - pred_days[mask]   # positivo = llegaste antes
    useful = (diff >= window[0]) & (diff <= window[1])
    return float(useful.mean())


def directional_error(pred_days, actual_days, mask):
    """
    Descompone el error en tardío (sin valor) y temprano (potencialmente útil).

    error > 0: pred > actual → predijiste más días de los que realmente tardó
               → el reminder habría llegado DESPUÉS del cargo → sin valor
    error ≤ 0: pred ≤ actual → predijiste igual o menos días
               → el reminder habría llegado ANTES del cargo → potencialmente útil
    """
    diff = pred_days[mask] - actual_days[mask]
    late_mask  = diff > 0
    early_mask = diff <= 0

    return {
        'late_rate':  float(late_mask.mean()),
        'early_rate': float(early_mask.mean()),
        'mae_late':   float(np.abs(diff[late_mask]).mean())  if late_mask.sum()  > 0 else 0.0,
        'mae_early':  float(np.abs(diff[early_mask]).mean()) if early_mask.sum() > 0 else 0.0,
    }


def coverage_at_k(pred_days, actual_days, mask, k_list=(3, 7, 14)):
    """
    % de predicciones con error absoluto ≤ k días.

    Coverage@3:  útil para reminders urgentes (T-1 a T-3)
    Coverage@7:  cubre el segmento ≤7d completo
    Coverage@14: cobertura del pipeline de reminders corto-plazo

    Complementa MAE: dos modelos con mismo MAE pueden tener distribuciones
    de error muy distintas — Coverage@K lo revela.
    """
    errors = np.abs(pred_days[mask] - actual_days[mask])
    return {f'coverage@{k}d': float((errors <= k).mean()) for k in k_list}


def expected_calibration_error(bucket_probs, y_bucket_true, n_bins=10):
    """
    Expected Calibration Error (ECE) del clasificador de buckets.

    Un modelo bien calibrado cumple: si P(bucket-j)=0.70 para un conjunto
    de predicciones, el 70% de esas predicciones realmente cae en bucket-j.

    ECE alto → las probabilidades no reflejan frecuencias reales →
    el expected-value decoder produce predicciones sistemáticamente sesgadas.

    Rango: 0.0 (perfectamente calibrado) — 1.0 (completamente descalibrado)
    Objetivo operacional: ECE < 0.10

    Fuente: Guo et al. (2017) "On Calibration of Modern Neural Networks"
    ICML 2017, Section 3.
    """
    confidences = bucket_probs.max(axis=1)
    predictions = bucket_probs.argmax(axis=1)

    # Solo evaluar sobre eventos observados (alineado con mask del test set)
    n = len(confidences)
    accuracies = (predictions == y_bucket_true).astype(float)

    bins = np.linspace(0.0, 1.0, n_bins + 1)
    ece = 0.0

    for i in range(n_bins):
        in_bin = (confidences >= bins[i]) & (confidences < bins[i + 1])
        if in_bin.sum() == 0:
            continue
        acc_bin  = accuracies[in_bin].mean()
        conf_bin = confidences[in_bin].mean()
        ece += (in_bin.sum() / n) * abs(acc_bin - conf_bin)

    return float(ece)


def concordance_index(pred_days, actual_days, mask, sample_size=50_000, seed=42):
    """
    Concordance Index (C-index): % de pares de usuarios correctamente
    ordenados por tiempo al próximo evento.

    C-index = 1.0 → el modelo ordena perfectamente quién pagará antes
    C-index = 0.5 → aleatorio, sin información de ranking
    C-index < 0.5 → peor que aleatorio

    Por qué importa para reminders:
        Si tienes presupuesto para N reminders, quieres enviárselos a los
        usuarios que van a intentar el cargo MÁS PRONTO. El C-index mide
        exactamente si el modelo los ordena bien. MAE no captura esto.

    Implementación vectorizada por bloques para O(n²) manejable con 50K muestra.
    Fuente: Harrell et al. (1982), JAMA — adaptado a TTE en pagos.
    """
    p = pred_days[mask]
    a = actual_days[mask]

    # Muestra reproducible para eficiencia
    if len(p) > sample_size:
        rng = np.random.default_rng(seed)
        idx = rng.choice(len(p), sample_size, replace=False)
        p, a = p[idx], a[idx]

    concordant = 0
    discordant = 0
    block = 1000

    for i in range(0, len(p), block):
        p_i = p[i:i + block]
        a_i = a[i:i + block]

        diff_a = a_i[:, None] - a[None, :]   # (block, N)
        diff_p = p_i[:, None] - p[None, :]

        comparable = diff_a != 0
        concordant += int(((diff_a > 0) == (diff_p > 0))[comparable].sum())
        discordant += int(((diff_a > 0) != (diff_p > 0))[comparable].sum())

    total = concordant + discordant
    return float(concordant / total) if total > 0 else 0.5


# ─────────────────────────────────────────────────────────────────────────────
# BLOQUE 2 — MÉTRICAS DE CASTIGO
# ─────────────────────────────────────────────────────────────────────────────

def asymmetric_mae(pred_days, actual_days, mask, alpha=2.0):
    """
    MAE asimétrico: penaliza errores tardíos alpha veces más que errores tempranos.

    error > 0 (tardío):   pred > actual → reminder llega después del cargo
                          costo = alpha × |error|
    error ≤ 0 (temprano): pred ≤ actual → reminder llega antes del cargo
                          costo = 1.0   × |error|

    alpha=2.0: llegar tarde cuesta el doble que llegar temprano.
    Ajustar alpha según el costo real de un retry vs una notificación extra.

    Fuente: Koenker & Bassett (1978) — Asymmetric loss, Econometrica.
    """
    errors = pred_days[mask] - actual_days[mask]
    weights = np.where(errors > 0, alpha, 1.0)
    return float((weights * np.abs(errors)).mean())


def pinball_loss(pred_days, actual_days, mask, quantile=0.90):
    """
    Quantile Loss (Pinball) al percentil objetivo.

    Es la función de pérdida que optimiza directamente el percentil que
    el GO/NO-GO gate evalúa (p90). Permite ver cuánto mejora cada
    iteración en ese quantile específico, no solo si pasa el threshold binario.

    Para q=0.90:
        Subestimar (pred < actual):  costo = 0.90 × |error|  — penaliza más
        Sobreestimar (pred > actual): costo = 0.10 × |error|  — penaliza menos

    Intuición para reminders: subestimar el tiempo al cargo (predecir muy pronto)
    es más tolerable que sobreestimar (predecir muy tarde → reminder inútil).

    Fuente: Koenker (2005) "Quantile Regression", Cambridge Univ. Press — ec. 1.2.
    """
    # error positivo = subestimación (pred < actual = llegaste antes de lo predicho)
    errors = actual_days[mask] - pred_days[mask]
    loss = np.where(
        errors >= 0,
        quantile * errors,
        (quantile - 1.0) * errors,
    )
    return float(loss.mean())


def business_cost_score(pred_days, actual_days, mask,
                        cost_late_per_day=1.0,
                        cost_early_per_day=0.1,
                        cost_missed=10.0,
                        actionable_window=(1, 14)):
    """
    Costo total simulado del sistema de reminders en unidades de negocio.

    3 tipos de resultado con sus costos:

        MISSED  (error > 1d tardío, dentro de ventana accionable):
            El reminder llegó después del cargo → sin efecto.
            Costo fijo: cost_missed (equivalente al costo de un retry rechazado).

        LATE    (0 < error ≤ 1d):
            Llegó tarde pero dentro de tolerancia de 1 día.
            Costo proporcional: cost_late_per_day × días de retraso.

        EARLY   (error ≤ 0):
            Llegó antes del cargo → potencialmente útil.
            Costo proporcional menor: cost_early_per_day × días de anticipación.
            (Demasiado temprano = fatiga de notificaciones, no es costo cero.)

    Defaults como índice relativo (cost_missed=10 unidades). Si tienes el
    costo real en pesos de un retry rechazado en SPIN, reemplaza cost_missed
    y el score total será el costo operativo estimado del período de test.
    """
    p = pred_days[mask]
    a = actual_days[mask]
    errors = p - a    # positivo = llegaste tarde

    tolerance     = 1.0
    missed_mask   = errors > tolerance
    late_mask     = (errors > 0) & (errors <= tolerance)
    early_mask    = errors <= 0

    # Solo contar como missed si el cargo estaba en la ventana accionable
    in_window     = (a >= actionable_window[0]) & (a <= actionable_window[1])

    cost_missed_t = float((missed_mask & in_window).sum() * cost_missed)
    cost_late_t   = float((np.abs(errors) * cost_late_per_day * late_mask).sum())
    cost_early_t  = float((np.abs(errors) * cost_early_per_day * early_mask).sum())
    total_cost    = cost_missed_t + cost_late_t + cost_early_t

    n             = int(mask.sum())

    return {
        'total_cost':       total_cost,
        'cost_per_event':   total_cost / n if n > 0 else 0.0,
        'pct_missed_actionable': float((missed_mask & in_window).mean()),
        'pct_missed_total':      float(missed_mask.mean()),
        'pct_late_marginal':     float(late_mask.mean()),
        'pct_late_total':        float((missed_mask | late_mask).mean()),
        'pct_early_useful': float((early_mask & in_window).mean()),
    }


# ─────────────────────────────────────────────────────────────────────────────
# BLOQUE 3 — EJECUCIÓN Y DISPLAY
# ─────────────────────────────────────────────────────────────────────────────

business_metrics_all = {}

for model_type, pred_days in predictions.items():

    print(f"\n{'='*80}")
    print(f"  {model_type} — MÉTRICAS DE NEGOCIO + CASTIGO")
    print(f"{'='*80}")

    # ── Bucket probs para ECE (inferencia rápida) ─────────────────────────────
    bp = predict_buckets(best_models[model_type], X_test, device, batch_size=8192)
    # ECE solo sobre eventos observados (alineado con mask)
    bp_obs         = bp[mask]
    y_bucket_obs   = seq_test.y_bucket[mask]

    # ── Calcular todas las métricas ───────────────────────────────────────────
    ewr_7d  = early_warning_rate(pred_days, actual_days, mask, window=(1, 7))
    ewr_14d = early_warning_rate(pred_days, actual_days, mask, window=(1, 14))
    dir_err = directional_error(pred_days, actual_days, mask)
    cov     = coverage_at_k(pred_days, actual_days, mask, k_list=(3, 7, 14))
    ece     = expected_calibration_error(bp_obs, y_bucket_obs, n_bins=10)

    print(f"\n  ⏳ Calculando C-index (muestra 50K)...")
    c_idx   = concordance_index(pred_days, actual_days, mask)

    asym    = asymmetric_mae(pred_days, actual_days, mask, alpha=2.0)
    pb_90   = pinball_loss(pred_days, actual_days, mask, quantile=0.90)
    pb_50   = pinball_loss(pred_days, actual_days, mask, quantile=0.50)
    bcs     = business_cost_score(
                  pred_days, actual_days, mask,
                  cost_late_per_day=1.0,
                  cost_early_per_day=0.1,
                  cost_missed=10.0,
              )

    # ── Display ───────────────────────────────────────────────────────────────
    print(f"\n  📊 MÉTRICAS DE NEGOCIO")
    print(f"  {'─'*60}")
    ewr_status  = '✅' if ewr_7d  >= 0.40 else '⚠️ '
    late_status = '✅' if dir_err['late_rate'] <= 0.30 else '⚠️ '
    cov7_status = '✅' if cov['coverage@7d']   >= 0.50 else '⚠️ '
    ece_status  = '✅' if ece    <= 0.10 else '⚠️ '
    ci_status   = '✅' if c_idx  >= 0.70 else '⚠️ '

    print(f"  {ewr_status}  Early Warning Rate (1-7d):   {ewr_7d*100:6.1f}%   "
          f"[objetivo ≥ 40%]  ← reminders útiles")
    print(f"       Early Warning Rate (1-14d):  {ewr_14d*100:6.1f}%")
    print(f"  {late_status}  Late Rate:                   {dir_err['late_rate']*100:6.1f}%   "
          f"[objetivo ≤ 30%]  ← sin valor de negocio")
    print(f"       Early Rate:                  {dir_err['early_rate']*100:6.1f}%")
    print(f"       MAE tardío:                  {dir_err['mae_late']:6.2f}d  "
          f"(α=2.0 → {dir_err['mae_late']*2:.2f}d ponderado)")
    print(f"       MAE temprano:                {dir_err['mae_early']:6.2f}d")
    print(f"  {cov7_status}  Coverage@3d: {cov['coverage@3d']*100:5.1f}%  "
          f"Coverage@7d: {cov['coverage@7d']*100:5.1f}%  "
          f"Coverage@14d: {cov['coverage@14d']*100:5.1f}%  [objetivo @7d ≥ 50%]")
    print(f"  {ece_status}  Calibration ECE:             {ece:7.4f}  "
          f"[objetivo < 0.10]")
    print(f"  {ci_status}  Concordance Index (C-index): {c_idx:.4f}  "
          f"[objetivo > 0.70]  ← calidad del ranking")

    print(f"\n  💰 MÉTRICAS DE CASTIGO")
    print(f"  {'─'*60}")
    print(f"  ⚖️   Asymmetric MAE (α=2.0):      {asym:7.2f}d  "
          f"← errores tardíos penalizados ×2")
    print(f"  🎯  Pinball Loss q=0.90:          {pb_90:7.4f}  "
          f"← penalización directa del p90")
    print(f"       Pinball Loss q=0.50:          {pb_50:7.4f}  "
          f"← penalización directa del p50")
    print(f"\n  📦  Business Cost Score (cost_missed=10 unidades):")
    print(f"       Costo total:          {bcs['total_cost']:>12,.1f} unidades")
    print(f"       Costo por evento:     {bcs['cost_per_event']:>12.4f} unidades/evento")
    print(f"       % Missed (actionable):  {bcs['pct_missed_actionable']*100:6.1f}%  ← tardío + dentro de ventana [1,14]d")
    print(f"       % Missed (total):       {bcs['pct_missed_total']*100:6.1f}%  ← todos los tardíos >1d")
    print(f"       % Late (marginal 0-1d): {bcs['pct_late_marginal']*100:6.1f}%")
    print(f"       % Late (total):         {bcs['pct_late_total']*100:6.1f}%  ← debe coincidir con Late Rate")
    print(f"       % Early útil (1-14d):   {bcs['pct_early_useful']*100:6.1f}%")

    # ── Guardar ───────────────────────────────────────────────────────────────
    business_metrics_all[model_type] = {
        'early_warning_rate_1_7d':   ewr_7d,
        'early_warning_rate_1_14d':  ewr_14d,
        'late_rate':                 dir_err['late_rate'],
        'early_rate':                dir_err['early_rate'],
        'mae_late':                  dir_err['mae_late'],
        'mae_early':                 dir_err['mae_early'],
        'coverage_3d':               cov['coverage@3d'],
        'coverage_7d':               cov['coverage@7d'],
        'coverage_14d':              cov['coverage@14d'],
        'calibration_ece':           ece,
        'concordance_index':         c_idx,
        'asymmetric_mae_alpha2':     asym,
        'pinball_loss_q90':          pb_90,
        'pinball_loss_q50':          pb_50,
        'business_cost_total':       bcs['total_cost'],
        'business_cost_per_event':   bcs['cost_per_event'],
        'pct_missed_actionable':     bcs['pct_missed_actionable'],
        'pct_missed_total':          bcs['pct_missed_total'],
        'pct_late_marginal':         bcs['pct_late_marginal'],
        'pct_late_total':            bcs['pct_late_total'],
        'pct_early_useful':          bcs['pct_early_useful'],
    }

# ── Comparativa final entre modelos ──────────────────────────────────────────
if len(business_metrics_all) > 1:
    print(f"\n\n{'='*80}")
    print("  COMPARATIVA DE MODELOS — MÉTRICAS DE NEGOCIO")
    print(f"{'='*80}")

    header_cols = [
        ('EWR@7d',    'early_warning_rate_1_7d',  lambda x: f"{x*100:.1f}%"),
        ('Late Rate', 'late_rate',                 lambda x: f"{x*100:.1f}%"),
        ('Cov@7d',    'coverage_7d',               lambda x: f"{x*100:.1f}%"),
        ('ECE',       'calibration_ece',            lambda x: f"{x:.4f}"),
        ('C-index',   'concordance_index',          lambda x: f"{x:.4f}"),
        ('AsymMAE',   'asymmetric_mae_alpha2',      lambda x: f"{x:.2f}d"),
        ('PB@90',     'pinball_loss_q90',            lambda x: f"{x:.4f}"),
        ('Cost/Ev.',  'business_cost_per_event',    lambda x: f"{x:.3f}"),
    ]

    # Header
    header = f"  {'Modelo':<8}" + "".join(f"  {c[0]:>9}" for c in header_cols)
    print(header)
    print("  " + "─" * (len(header) - 2))

    for mt, m in business_metrics_all.items():
        row = f"  {mt:<8}" + "".join(f"  {c[2](m[c[1]]):>9}" for c in header_cols)
        print(row)

    # Mejor modelo por métrica
    print(f"\n  Mejor por métrica:")
    objectives = {
        'EWR@7d':    ('early_warning_rate_1_7d', max),
        'Late Rate': ('late_rate',                min),
        'Cov@7d':    ('coverage_7d',              max),
        'ECE':       ('calibration_ece',           min),
        'C-index':   ('concordance_index',         max),
        'AsymMAE':   ('asymmetric_mae_alpha2',     min),
        'PB@90':     ('pinball_loss_q90',           min),
    }
    for label, (key, fn) in objectives.items():
        best = fn(business_metrics_all.items(), key=lambda x: x[1][key])
        print(f"    {label:<12}: {best[0]}")

# ── Guardar JSON ──────────────────────────────────────────────────────────────
metrics_path = OUTPUT_DIR / 'business_metrics.json'
with open(metrics_path, 'w') as f:
    json.dump(business_metrics_all, f, indent=2)
print(f"\n💾 Métricas guardadas: {metrics_path}")



  GRU — MÉTRICAS DE NEGOCIO + CASTIGO

  ⏳ Calculando C-index (muestra 50K)...

  📊 MÉTRICAS DE NEGOCIO
  ────────────────────────────────────────────────────────────
  ⚠️   Early Warning Rate (1-7d):     34.7%   [objetivo ≥ 40%]  ← reminders útiles
       Early Warning Rate (1-14d):    49.1%
  ✅  Late Rate:                     15.4%   [objetivo ≤ 30%]  ← sin valor de negocio
       Early Rate:                    84.6%
       MAE tardío:                    2.95d  (α=2.0 → 5.90d ponderado)
       MAE temprano:                  6.98d
  ✅  Coverage@3d:  52.1%  Coverage@7d:  69.9%  Coverage@14d:  85.0%  [objetivo @7d ≥ 50%]
  ⚠️   Calibration ECE:              0.1122  [objetivo < 0.10]
  ✅  Concordance Index (C-index): 0.7926  [objetivo > 0.70]  ← calidad del ranking

  💰 MÉTRICAS DE CASTIGO
  ────────────────────────────────────────────────────────────
  ⚖️   Asymmetric MAE (α=2.0):         6.82d  ← errores tardíos penalizados ×2
  🎯  Pinball Loss q=0.90:           5.3604  ← penalización

## 17. Temperature Scaling + Persistencia (16b)

Calibracion post-hoc. **Agregado v1.9.0:** se persiste el dict `temperatures` a disco. Antes solo vivia en RAM, lo que hacia imposible la inferencia de kernel fresco sin re-entrenar.

In [21]:
# ═════════════════════════════════════════════════════════════════════════════
# TEMPERATURE SCALING — Post-hoc calibration sin reentrenar
# Ref: Guo et al. (2017) ICML
# ═════════════════════════════════════════════════════════════════════════════

from scipy.optimize import minimize_scalar

def find_temperature(logits_val, y_val):
    """Encuentra T óptimo minimizando NLL sobre validation."""
    def nll(T):
        # Scale logits by 1/T, then compute cross-entropy
        scaled = logits_val / T
        # Stable softmax
        shifted = scaled - scaled.max(axis=1, keepdims=True)
        exp_s = np.exp(shifted)
        probs = exp_s / exp_s.sum(axis=1, keepdims=True)
        # NLL
        eps = 1e-10
        log_probs = np.log(probs[np.arange(len(y_val)), y_val] + eps)
        return -log_probs.mean()
    
    result = minimize_scalar(nll, bounds=(0.1, 10.0), method='bounded')
    return result.x

def calibrated_probs(logits, T):
    """Aplica temperature scaling a logits."""
    scaled = logits / T
    shifted = scaled - scaled.max(axis=1, keepdims=True)
    exp_s = np.exp(shifted)
    return exp_s / exp_s.sum(axis=1, keepdims=True)

@torch.no_grad()
def extract_logits(model, X, device, batch_size=8192):
    """Extrae logits raw (antes de softmax)."""
    model = model.to(device)
    model.eval()
    all_logits = []
    for start in range(0, len(X), batch_size):
        end = min(start + batch_size, len(X))
        X_batch = torch.from_numpy(X[start:end]).float().to(device)
        logits_bucket, _, _ = model(X_batch)
        all_logits.append(logits_bucket.cpu().numpy())
        del X_batch
    return np.concatenate(all_logits, axis=0)


print(f"\n{'='*95}")
print(f"  TEMPERATURE SCALING — Calibración post-hoc")
print(f"{'='*95}")

calibrated_probs_all = {}
temperatures = {}

for mt, model in best_models.items():
    # Extraer logits de validation
    logits_val = extract_logits(model, X_val, device)
    y_val_buckets = seq_val.y_bucket
    
    # Encontrar T óptimo
    T_opt = find_temperature(logits_val, y_val_buckets)
    temperatures[mt] = T_opt
    
    # Aplicar a test
    logits_test = extract_logits(model, X_test, device)
    cal_probs = calibrated_probs(logits_test, T_opt)
    calibrated_probs_all[mt] = cal_probs
    
    # Calcular ECE antes y después
    bp_obs = bucket_probs_all[mt][mask]
    cal_obs = cal_probs[mask]
    y_bucket_obs = seq_test.y_bucket[mask]
    
    ece_before = expected_calibration_error(bp_obs, y_bucket_obs, n_bins=10)
    ece_after = expected_calibration_error(cal_obs, y_bucket_obs, n_bins=10)
    
    print(f"\n  {mt}: T={T_opt:.3f}")
    print(f"    ECE antes:  {ece_before:.4f}")
    print(f"    ECE después:{ece_after:.4f}  {'✅' if ece_after < 0.10 else '⚠️'}  [objetivo < 0.10]")
    print(f"    Δ ECE:      {ece_before - ece_after:+.4f}")

# Recalcular predicciones con probs calibradas
print(f"\n  Recalculando predicciones con probs calibradas...")
predictions_calibrated = {}
for mt in best_models:
    q = rec_decoder_q if rec_decoder_q else 0.35
    predictions_calibrated[mt] = decode_quantile(calibrated_probs_all[mt], CONFIG, q=q)
    r = compute_full_metrics(predictions_calibrated[mt], actual_days, mask)
    print(f"    {mt} (q={q}, calibrado): MAE={r['mae']:.2f}d, Late={r['late%']*100:.1f}%, "
          f"EWR@7d={r['ewr7']*100:.1f}%")

if torch.cuda.is_available():
    torch.cuda.empty_cache()


# --- Persistencia de temperatures (ADD v1.9.0) ---
# Gap operativo: el dict temperatures solo vivia en RAM. Sin esto, la
# inferencia de kernel fresco era imposible sin re-correr el training.
_temp_path = OUTPUT_DIR / 'temperatures.joblib'
joblib.dump({k: float(v) for k, v in temperatures.items()}, _temp_path)
print(f'  temperatures persistido: {_temp_path}')


  TEMPERATURE SCALING — Calibración post-hoc

  GRU: T=1.470
    ECE antes:  0.1122
    ECE después:0.0609  ✅  [objetivo < 0.10]
    Δ ECE:      +0.0513

  RNN: T=1.366
    ECE antes:  0.0814
    ECE después:0.0391  ✅  [objetivo < 0.10]
    Δ ECE:      +0.0423

  LSTM: T=1.515
    ECE antes:  0.1142
    ECE después:0.0669  ✅  [objetivo < 0.10]
    Δ ECE:      +0.0473

  TCN: T=2.007
    ECE antes:  0.2303
    ECE después:0.1174  ⚠️  [objetivo < 0.10]
    Δ ECE:      +0.1129

  DEEP_GRU: T=1.594
    ECE antes:  0.1471
    ECE después:0.0441  ✅  [objetivo < 0.10]
    Δ ECE:      +0.1029

  Recalculando predicciones con probs calibradas...
    GRU (q=0.35, calibrado): MAE=6.70d, Late=15.6%, EWR@7d=31.6%
    RNN (q=0.35, calibrado): MAE=7.08d, Late=13.9%, EWR@7d=28.5%
    LSTM (q=0.35, calibrado): MAE=6.66d, Late=16.7%, EWR@7d=30.2%
    TCN (q=0.35, calibrado): MAE=7.81d, Late=18.6%, EWR@7d=27.9%
    DEEP_GRU (q=0.35, calibrado): MAE=6.42d, Late=16.7%, EWR@7d=33.0%
  temperatures persisti

## 18. Ensemble Calibrado + Grid Search Integrado (16f) — Fuente de Verdad

Produce `predictions_final`, `bp_final`, `best_model_type_final`, `best_q_final`. **Agregado v1.9.0:** `final_selection` se persiste a JSON.

In [22]:
# =============================================================================
# CELDA 16f -- ENSEMBLE CALIBRADO + GRID SEARCH INTEGRADO
# FUENTE DE VERDAD: produce predictions_final, bp_final, best_model_type_final
# v1.5.7: Fix ECE/C-index reporting cuando modelo individual gana vs ensemble
# =============================================================================

print(f"\n{'='*95}")
print(f"  ENSEMBLE CALIBRADO + GRID SEARCH INTEGRADO")
print(f"{'='*95}")

# --- PASO 1: Calibrar probs de VALIDATION para cada modelo -------------------
print(f"\n  PASO 1 -- Calibrar probs de validation con T ya optimizados...")
calibrated_probs_val = {}

for mt, model in best_models.items():
    logits_v = extract_logits(model, X_val, device)
    cal_probs_v = calibrated_probs(logits_v, temperatures[mt])
    calibrated_probs_val[mt] = cal_probs_v
    print(f"    {mt}: shape {cal_probs_v.shape} (T={temperatures[mt]:.3f})")

# --- Validation ground truth (previously in 16d, now hoisted here) -----------
# Required for grid search on val. 16d is superseded; 16f owns this now.
actual_val = seq_val.y_true_tte
mask_val   = (seq_val.y_true_event_observed == 1) & ~np.isnan(seq_val.y_true_tte)
print(f"  Val ground truth: {mask_val.sum():,} observed / {len(mask_val):,} total")

# --- PASO 2: Grid search sobre probs CALIBRADAS de validation ----------------
print(f"\n  PASO 2 -- Grid search sobre probs calibradas (Late% <= 30%, max EWR@7d)...")

q_grid = np.arange(0.15, 0.55, 0.01)
optimal_q_cal = {}
optimal_q_cal_test = {}

for mt in CONFIG['models_to_train']:
    bp_cal_v = calibrated_probs_val[mt]
    bp_cal_t = calibrated_probs_all[mt]

    best_ewr = -1.0
    best_q   = 0.35
    best_r_v = None

    for q in q_grid:
        pred_v = decode_quantile(bp_cal_v, CONFIG, q=q)
        r_v = compute_full_metrics(pred_v, actual_val, mask_val)
        if r_v['late%'] <= 0.30 and r_v['ewr7'] > best_ewr:
            best_ewr = r_v['ewr7']
            best_q   = float(round(q, 2))
            best_r_v = r_v

    optimal_q_cal[mt] = best_q

    # Confirmar en test con probs calibradas (lectura, no selección)
    pred_t = decode_quantile(bp_cal_t, CONFIG, q=best_q)
    r_t    = compute_full_metrics(pred_t, actual_days, mask)
    optimal_q_cal_test[mt] = r_t

    print(f"\n    {mt}: q*={best_q:.2f}")
    if best_r_v:
        print(f"      Val  (cal): EWR@7d={best_r_v['ewr7']*100:.1f}%  Late={best_r_v['late%']*100:.1f}%  MAE={best_r_v['mae']:.2f}d")
    print(f"      Test (cal): EWR@7d={r_t['ewr7']*100:.1f}%  Late={r_t['late%']*100:.1f}%  MAE={r_t['mae']:.2f}d  AsymMAE={r_t['asym']:.2f}d")

# --- PASO 3: Construir ensemble calibrado ------------------------------------
print(f"\n  PASO 3 -- Ensemble (promedio de probs calibradas de test)...")
ensemble_probs_cal = np.mean(
    [calibrated_probs_all[mt] for mt in CONFIG['models_to_train']], axis=0
)
print(f"    Shape: {ensemble_probs_cal.shape}")
print(f"    Suma probs check: min={ensemble_probs_cal.sum(axis=1).min():.6f}, max={ensemble_probs_cal.sum(axis=1).max():.6f}")

# --- PASO 4: Grid search sobre ensemble calibrado (val) ----------------------
print(f"\n  PASO 4 -- Grid search q* del ensemble calibrado (val)...")
ensemble_probs_cal_val = np.mean(
    [calibrated_probs_val[mt] for mt in CONFIG['models_to_train']], axis=0
)

best_ewr_ens = -1.0
best_q_ens   = 0.35
best_r_ens_v = None

for q in q_grid:
    pred_ens_v = decode_quantile(ensemble_probs_cal_val, CONFIG, q=q)
    r_ens_v    = compute_full_metrics(pred_ens_v, actual_val, mask_val)
    if r_ens_v['late%'] <= 0.30 and r_ens_v['ewr7'] > best_ewr_ens:
        best_ewr_ens = r_ens_v['ewr7']
        best_q_ens   = float(round(q, 2))
        best_r_ens_v = r_ens_v

if best_r_ens_v:
    print(f"    Ensemble q*={best_q_ens:.2f}")
    print(f"      Val: EWR@7d={best_r_ens_v['ewr7']*100:.1f}%  Late={best_r_ens_v['late%']*100:.1f}%  MAE={best_r_ens_v['mae']:.2f}d")

# --- PASO 5: Evaluacion final del ensemble en test ---------------------------
print(f"\n  PASO 5 -- Evaluacion ensemble en test set...")
pred_ens_test = decode_quantile(ensemble_probs_cal, CONFIG, q=best_q_ens)
r_ens_test    = compute_full_metrics(pred_ens_test, actual_days, mask)

ece_ens = expected_calibration_error(ensemble_probs_cal[mask], seq_test.y_bucket[mask])
print(f"  Calculando C-index ensemble (muestra 50K)...")
cindex_ens = concordance_index(pred_ens_test, actual_days, mask)

# --- TABLA COMPARATIVA -------------------------------------------------------
print(f"\n\n{'='*95}")
print(f"  COMPARATIVA: Modelos individuales calibrados vs ENSEMBLE CALIBRADO")
print(f"{'='*95}")
print(f"  {'Fuente':<22} {'q*':>5} {'MAE':>6} {'p90':>6} {'Late%':>7} {'EWR@7d':>7} {'AsymMAE':>8} {'ECE':>7}")
print(f"  {'-'*72}")

for mt in CONFIG['models_to_train']:
    r    = optimal_q_cal_test[mt]
    q_v  = optimal_q_cal[mt]
    ece_m = expected_calibration_error(calibrated_probs_all[mt][mask], seq_test.y_bucket[mask])
    ewr_ok  = '|OK' if r['ewr7']  >= 0.40 else '|--'
    late_ok = '|OK' if r['late%'] <= 0.30 else '|--'
    print(f"  {mt:<22} {q_v:>5.2f} {r['mae']:>5.2f}d {r['p90e']:>5.2f}d "
          f"{r['late%']*100:>6.1f}%{late_ok}"
          f"{r['ewr7']*100:>6.1f}%{ewr_ok}"
          f"{r['asym']:>7.2f}d {ece_m:>6.4f}")

print(f"  {'-'*72}")
ewr_ok  = '|OK' if r_ens_test['ewr7']  >= 0.40 else '|--'
late_ok = '|OK' if r_ens_test['late%'] <= 0.30 else '|--'
print(f"  {'[ENSEMBLE CAL x4]':<22} {best_q_ens:>5.2f} {r_ens_test['mae']:>5.2f}d {r_ens_test['p90e']:>5.2f}d "
      f"{r_ens_test['late%']*100:>6.1f}%{late_ok}"
      f"{r_ens_test['ewr7']*100:>6.1f}%{ewr_ok}"
      f"{r_ens_test['asym']:>7.2f}d {ece_ens:>6.4f}")

# --- Seleccionar mejor resultado (ensemble vs mejor individual) ---------------
# FIX v1.5.7: ECE y C-index se reportan del modelo ELEGIDO, no siempre del ensemble
best_individual_ewr = max(r['ewr7'] for r in optimal_q_cal_test.values())
best_individual_mt  = max(optimal_q_cal_test, key=lambda k: optimal_q_cal_test[k]['ewr7'])

if r_ens_test['ewr7'] >= best_individual_ewr:
    predictions_final     = pred_ens_test
    best_model_type_final = 'ENSEMBLE_CAL'
    best_q_final          = best_q_ens
    r_final               = r_ens_test
    bp_final              = ensemble_probs_cal   # <-- FIX v1.5.7
    ece_final             = ece_ens
    cindex_final          = cindex_ens
    print(f"\n  Mejor: ENSEMBLE calibrado (q={best_q_ens:.2f})")
else:
    best_q_ind            = optimal_q_cal[best_individual_mt]
    bp_final              = calibrated_probs_all[best_individual_mt]  # <-- FIX v1.5.7
    predictions_final     = decode_quantile(bp_final, CONFIG, q=best_q_ind)
    best_model_type_final = f'{best_individual_mt}_CAL'
    best_q_final          = best_q_ind
    r_final               = optimal_q_cal_test[best_individual_mt]
    ece_final             = expected_calibration_error(bp_final[mask], seq_test.y_bucket[mask])
    pred_ev_ind           = decode_expected_value(bp_final, CONFIG)
    cindex_final          = concordance_index(pred_ev_ind, actual_days, mask)
    print(f"\n  Mejor individual: {best_individual_mt} calibrado (q={best_q_ind:.2f})")

# FIX v1.5.7: Usar ece_final/cindex_final del modelo elegido, no hardcoded ensemble
print(f"\n  == RESULTADO FINAL (modelo: {best_model_type_final}) ==")
print(f"  EWR@7d:    {r_final['ewr7']*100:.1f}%  {'OK' if r_final['ewr7'] >= 0.40 else '--'}  [>= 40%]")
print(f"  Late Rate: {r_final['late%']*100:.1f}%  {'OK' if r_final['late%'] <= 0.30 else '--'}  [<= 30%]")
print(f"  MAE:       {r_final['mae']:.2f}d")
print(f"  p90:       {r_final['p90e']:.2f}d  {'OK' if r_final['p90e'] < 12.0 else '--'}  [< 12d]")
print(f"  AsymMAE:   {r_final['asym']:.2f}d")
print(f"  ECE:       {ece_final:.4f}  {'OK' if ece_final < 0.10 else '--'}  [< 0.10]")
print(f"  C-index:   {cindex_final:.4f}  {'OK' if cindex_final >= 0.70 else '--'}  [> 0.70]")

# --- ARTIFACT: final_selection (única fuente de verdad) ----------------------
final_selection = {
    'best_model_type_final': best_model_type_final,
    'best_q_final': best_q_final,
    'decoder_policy': f'quantile q={best_q_final:.2f} para timing, EV para C-index',
    'temperature': temperatures.get(best_individual_mt, None) if best_model_type_final != 'ENSEMBLE_CAL' else {k: v for k, v in temperatures.items()},
    'ece_final': float(ece_final),
    'cindex_final': float(cindex_final),
    'r_final': {k: float(v) if isinstance(v, (float, np.floating)) else v for k, v in r_final.items()},
}
print(f"\n  final_selection artifact: {json.dumps({k: str(v)[:50] for k,v in final_selection.items()}, indent=2)}")

if torch.cuda.is_available():
    torch.cuda.empty_cache()


# --- Persistencia de final_selection (ADD v1.9.0) ---
_fs_path = OUTPUT_DIR / 'final_selection_v1_9_0.json'
with open(_fs_path, 'w') as _f:
    json.dump(final_selection, _f, indent=2, default=str)
print(f'  final_selection persistido: {_fs_path}')


  ENSEMBLE CALIBRADO + GRID SEARCH INTEGRADO

  PASO 1 -- Calibrar probs de validation con T ya optimizados...
    GRU: shape (828438, 10) (T=1.470)
    RNN: shape (828438, 10) (T=1.366)
    LSTM: shape (828438, 10) (T=1.515)
    TCN: shape (828438, 10) (T=2.007)
    DEEP_GRU: shape (828438, 10) (T=1.594)
  Val ground truth: 792,841 observed / 828,438 total

  PASO 2 -- Grid search sobre probs calibradas (Late% <= 30%, max EWR@7d)...

    GRU: q*=0.15
      Val  (cal): EWR@7d=29.8%  Late=5.7%  MAE=9.80d
      Test (cal): EWR@7d=33.1%  Late=6.8%  MAE=8.56d  AsymMAE=8.67d

    RNN: q*=0.15
      Val  (cal): EWR@7d=30.1%  Late=5.5%  MAE=10.06d
      Test (cal): EWR@7d=33.5%  Late=6.4%  MAE=8.88d  AsymMAE=9.00d

    LSTM: q*=0.15
      Val  (cal): EWR@7d=29.4%  Late=5.8%  MAE=9.73d
      Test (cal): EWR@7d=32.7%  Late=6.8%  MAE=8.57d  AsymMAE=8.67d

    TCN: q*=0.15
      Val  (cal): EWR@7d=32.2%  Late=4.3%  MAE=11.77d
      Test (cal): EWR@7d=36.0%  Late=4.8%  MAE=10.48d  AsymMAE=10.53d


## 19. Abstention Policy (16g)

**Fix P1-4.** El 16g original dejaba dos politicas (LogReg y P(bucket_0)) conviviendo, usadas inconsistentemente downstream. Aqui se entrena la LogReg con metodologia correcta (fit:train / tune:val / eval:test), se compara con el proxy P(bucket_0), y se elige **una sola** `abstention_mask` (la de mayor AUC en validation), usada de forma consistente por las Action Cards.

**Riesgo abierto declarado:** ambas politicas tienen AUC bajo en test. La abstencion no es confiable todavia; es un componente debil conocido.

In [23]:
# =============================================================================
# CELDA 16g -- ABSTENTION POLICY REDISENIADA v1.5.7
# FIX CRITICO: fit en TRAIN, threshold en VAL, evaluacion locked en TEST
# v1.5.6 entrenaba en val y tuneaba en test — incorrecto metodologicamente
# =============================================================================

from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_auc_score
import warnings, pickle

print(f"\n{'='*95}")
print(f"  ABSTENTION POLICY v1.5.7 -- fit:train | tune:val | eval:test")
print(f"{'='*95}")

# --- PASO 1: Feature engineering para el clasificador ------------------------
def build_censoring_features(X_seq):
    """
    Agrega la secuencia temporal a features tabulares para LR.
    X_seq: (N, lookback, n_features) -> (N, 2*n_features) [ultimo | media]
    """
    last_step = X_seq[:, -1, :]     # estado mas reciente
    mean_step = X_seq.mean(axis=1)  # promedio historico
    return np.concatenate([last_step, mean_step], axis=1)

# FIX v1.5.7: Construir features para TRAIN, VAL y TEST
X_clf_train = build_censoring_features(X_train)
X_clf_val   = build_censoring_features(X_val)
X_clf_test  = build_censoring_features(X_test)

y_clf_train = seq_train.y_true_event_observed.astype(int)
y_clf_val   = seq_val.y_true_event_observed.astype(int)
y_clf_test  = seq_test.y_true_event_observed.astype(int)

n_cen_train = int((y_clf_train == 0).sum())
n_cen_val   = int((y_clf_val == 0).sum())
n_cen_test  = int((y_clf_test == 0).sum())
print(f"\n  Dataset:")
print(f"    Train: {len(y_clf_train):,} total | {n_cen_train:,} censurados ({n_cen_train/len(y_clf_train)*100:.1f}%)")
print(f"    Val:   {len(y_clf_val):,} total | {n_cen_val:,} censurados ({n_cen_val/len(y_clf_val)*100:.1f}%)")
print(f"    Test:  {len(y_clf_test):,} total | {n_cen_test:,} censurados ({n_cen_test/len(y_clf_test)*100:.1f}%)")

# --- PASO 2: fit en TRAIN (FIX v1.5.7 — antes era val) ----------------------
print(f"\n  PASO 2 -- fit LogisticRegression en TRAIN (target=event_observed)...")

scaler_clf    = StandardScaler()
X_clf_train_s = scaler_clf.fit_transform(X_clf_train)
X_clf_val_s   = scaler_clf.transform(X_clf_val)
X_clf_test_s  = scaler_clf.transform(X_clf_test)

clf = LogisticRegression(
    max_iter=300, class_weight='balanced',
    C=1.0, solver='lbfgs', random_state=42, n_jobs=-1,
)
with warnings.catch_warnings():
    warnings.simplefilter('ignore')
    clf.fit(X_clf_train_s, y_clf_train)

print(f"  Entrenado sobre {len(y_clf_train):,} muestras, {X_clf_train_s.shape[1]} features")

# --- PASO 3: Threshold tuning en VAL (FIX v1.5.7 — antes era test) ----------
print(f"\n  PASO 3 -- Threshold tuning en VALIDATION:")
p_event_val  = clf.predict_proba(X_clf_val_s)[:, 1]
p_censor_val = 1.0 - p_event_val

auc_val = roc_auc_score(y_clf_val, p_event_val)
print(f"  AUC-ROC val: {auc_val:.4f}")

actually_censored_val = (y_clf_val == 0)
best_thresh_lr = 0.3
best_f1_lr     = -1.0

print(f"\n  {'Thresh':>7} {'Abstain%':>9} {'Precision':>10} {'Recall':>8} {'F1':>7}")
print(f"  {'-'*50}")

for thresh in [0.20, 0.30, 0.40, 0.50, 0.60, 0.70]:
    abstain    = p_censor_val > thresh
    n_abstain  = int(abstain.sum())
    pct_abs    = n_abstain / len(p_censor_val) * 100

    prec = float((abstain & actually_censored_val).sum() / max(n_abstain, 1))
    rec  = float((abstain & actually_censored_val).sum() / max(actually_censored_val.sum(), 1))
    f1   = 2*prec*rec/(prec+rec+1e-10)

    if f1 > best_f1_lr:
        best_f1_lr     = f1
        best_thresh_lr = thresh

    print(f"  {thresh:>7.2f} {pct_abs:>8.1f}% {prec:>9.3f}  {rec:>7.3f}  {f1:>6.4f}")

print(f"\n  Threshold optimo (val, F1): {best_thresh_lr:.2f}")

# --- PASO 4: Evaluacion LOCKED en TEST --------------------------------------
print(f"\n  PASO 4 -- Evaluacion locked en TEST (threshold={best_thresh_lr:.2f} congelado):")
p_event_test  = clf.predict_proba(X_clf_test_s)[:, 1]
p_censor_test = 1.0 - p_event_test

auc_test = roc_auc_score(y_clf_test, p_event_test)
actually_censored_test = (y_clf_test == 0)
abstain_test = p_censor_test > best_thresh_lr

prec_test = float((abstain_test & actually_censored_test).sum() / max(abstain_test.sum(), 1))
rec_test  = float((abstain_test & actually_censored_test).sum() / max(actually_censored_test.sum(), 1))
f1_test   = 2*prec_test*rec_test/(prec_test+rec_test+1e-10)

print(f"  AUC-ROC test: {auc_test:.4f}")
print(f"  Precision:    {prec_test:.3f}")
print(f"  Recall:       {rec_test:.3f}")
print(f"  F1:           {f1_test:.4f}")
print(f"  Abstain%:     {abstain_test.mean()*100:.1f}%")

# Metricas del modelo principal sobre usuarios NO abstenidos
send_mask = ~abstain_test & mask
if send_mask.sum() > 10:
    err_s   = np.abs(predictions_final[send_mask] - actual_days[send_mask])
    signed_s = predictions_final[send_mask] - actual_days[send_mask]
    diff_s   = actual_days[send_mask] - predictions_final[send_mask]
    mae_s    = float(err_s.mean())
    late_s   = float((signed_s > 0).mean())
    ewr_s    = float(((diff_s >= 1) & (diff_s <= 7)).mean())
    print(f"\n  Metricas post-abstention (test, thresh={best_thresh_lr:.2f}):")
    print(f"    MAE:    {mae_s:.2f}d")
    print(f"    Late%:  {late_s*100:.1f}%")
    print(f"    EWR@7d: {ewr_s*100:.1f}%")

# --- PASO 5: Comparacion LR vs bucket-0 policy -------------------------------

# FIX P0 R4: LogReg abstention tiene AUC 0.51 en test (roto).
# Reemplazo: usar P(bucket_0) del modelo como proxy.
# bucket_0 = "no event in window" en el encoding actual.

best_bp = bucket_probs_all[best_model_type]
p_censor_model = best_bp[:, 0]  # P(bucket_0) directo

# Grid search threshold sobre validation
from sklearn.metrics import roc_auc_score
_val_bp = {mt: predict_full(best_models[mt], X_val, device)[0] for mt in best_models}
_p_val = _val_bp[best_model_type][:, 0]
_y_val = (seq_val.y_true_event_observed == 0).astype(int)

auc_model_proxy = roc_auc_score(_y_val, _p_val)
print(f"  AUC P(bucket_0) en val: {auc_model_proxy:.4f}")
print(f"  (compara vs LogReg que tenia AUC test=0.51)")

# Threshold óptimo: maximizar balanced accuracy
from sklearn.metrics import balanced_accuracy_score
thresholds = np.linspace(0.1, 0.9, 41)
scores = [balanced_accuracy_score(_y_val, (_p_val > t).astype(int)) for t in thresholds]
best_thresh_model = thresholds[np.argmax(scores)]
print(f"  Threshold optimo: {best_thresh_model:.3f} (balanced_acc={max(scores):.3f})")

# Aplicar a test
_abstain_b0 = p_censor_model > best_thresh_model
print(f"  Abstenciones en test (P(bucket_0)): {_abstain_b0.sum():,} ({_abstain_b0.mean()*100:.1f}%)")


print(f"\n  COMPARACION: LR policy vs bucket-0:")
bp_gru_for_comparison = calibrated_probs_all.get('GRU', list(calibrated_probs_all.values())[0])
p_b0    = bp_gru_for_comparison[:, 0]
abstain_b0  = p_b0 > 0.3
prec_b0 = float((abstain_b0 & actually_censored_test).sum() / max(abstain_b0.sum(), 1))
rec_b0  = float((abstain_b0 & actually_censored_test).sum() / max(actually_censored_test.sum(), 1))
f1_b0   = 2*prec_b0*rec_b0/(prec_b0+rec_b0+1e-10)

print(f"  {'Policy':<22} {'Abstain%':>9} {'Precision':>10} {'Recall':>8} {'F1':>7}")
print(f"  {'-'*60}")
print(f"  {'bucket-0 (deprecated)':<22} {abstain_b0.mean()*100:>8.1f}% {prec_b0:>9.3f}  {rec_b0:>7.3f}  {f1_b0:>6.4f}")
print(f"  {'LogReg v1.5.7':<22} {abstain_test.mean()*100:>8.1f}% {prec_test:>9.3f}  {rec_test:>7.3f}  {f1_test:>6.4f}")

# --- PASO 6: Funcion reusable para produccion --------------------------------
def score_censoring_policy(X_seq_new, scaler=scaler_clf, model=clf, threshold=best_thresh_lr):
    """
    Función de producción para la abstention policy.
    
    Args:
        X_seq_new: np.ndarray (N, lookback, n_features) — secuencias nuevas
        scaler: StandardScaler fitted (de 16g)
        model: LogisticRegression fitted (de 16g)
        threshold: float — threshold congelado de val
    
    Returns:
        p_censor: np.ndarray (N,) — probabilidad de censoring
        abstention_mask: np.ndarray (N,) bool — True = no enviar reminder
    """
    feats = build_censoring_features(X_seq_new)
    feats_s = scaler.transform(feats)
    p_event = model.predict_proba(feats_s)[:, 1]
    p_censor = 1.0 - p_event
    abstention_mask = p_censor > threshold
    return p_censor, abstention_mask

print(f"\n  score_censoring_policy() disponible para inferencia downstream.")

# --- Guardar clasificador para produccion ---
clf_path = OUTPUT_DIR / 'censoring_classifier_v1_9_0.pkl'
with open(clf_path, 'wb') as f_pkl:
    pickle.dump({
        'clf': clf, 'scaler': scaler_clf,
        'threshold': best_thresh_lr,
        'auc_roc_val': auc_val,
        'auc_roc_test': auc_test,
        'trained_on': 'train',           # FIX v1.5.7: documenta split de entrenamiento
        'threshold_tuned_on': 'val',     # FIX v1.5.7: documenta split de tuning
        'n_features': X_clf_train_s.shape[1],
        'feature_names': HMM_FEATURES,
        'version': 'v1.9.0',
    }, f_pkl)
print(f"\n  Clasificador guardado: {clf_path}")
print(f"  NOTA: fit=train, threshold=val, eval=test (correcto para shipping)")


# === CONSOLIDACION v1.9.0 — una sola abstention policy (fix P1-4) ===
# Se elige la politica con mayor AUC en VALIDATION y se expone de forma
# unica. Downstream (Action Cards) usa SOLO esta 'abstention_mask'.
if auc_model_proxy >= auc_val:
    abstention_policy    = 'p_bucket0'
    abstention_threshold = float(best_thresh_model)
    abstention_mask      = (p_censor_model > best_thresh_model)
else:
    abstention_policy    = 'logreg'
    abstention_threshold = float(best_thresh_lr)
    abstention_mask      = (p_censor_test > best_thresh_lr)

print(f"\n  Politica de abstencion ELEGIDA: {abstention_policy}")
print(f"    threshold = {abstention_threshold:.3f}")
print(f"    abstain%  = {abstention_mask.mean()*100:.1f}%")
print(f"  RIESGO ABIERTO: AUC bajo en test en ambas politicas.")
print(f"  La abstencion NO es confiable todavia — componente debil conocido.")


  ABSTENTION POLICY v1.5.7 -- fit:train | tune:val | eval:test

  Dataset:
    Train: 3,939,761 total | 141,328 censurados (3.6%)
    Val:   828,438 total | 35,597 censurados (4.3%)
    Test:  920,799 total | 181,026 censurados (19.7%)

  PASO 2 -- fit LogisticRegression en TRAIN (target=event_observed)...
  Entrenado sobre 3,939,761 muestras, 94 features

  PASO 3 -- Threshold tuning en VALIDATION:
  AUC-ROC val: 0.6888

   Thresh  Abstain%  Precision   Recall      F1
  --------------------------------------------------
     0.20     85.6%     0.048    0.952  0.0910
     0.30     70.2%     0.053    0.859  0.0990
     0.40     54.6%     0.059    0.747  0.1089
     0.50     41.8%     0.067    0.654  0.1220
     0.60     31.4%     0.078    0.574  0.1381
     0.70     21.6%     0.094    0.475  0.1573

  Threshold optimo (val, F1): 0.70

  PASO 4 -- Evaluacion locked en TEST (threshold=0.70 congelado):
  AUC-ROC test: 0.5665
  Precision:    0.224
  Recall:       0.291
  F1:           0.25

## 20. Gate GO/NO-GO Oficial (16j) + Gate Competitivo

**Fix P0-6.** El gate original podia decir GO con thresholds absolutos mientras el modelo perdia contra un predictor constante. Aqui el gate tiene dos partes y la decision final exige **ambas**:

1. **Gate absoluto:** MAE, p90, Late%, EWR@7d, ECE, C-index vs thresholds.
2. **Gate competitivo:** el modelo debe ganarle a `np.full(N, mediana)` en MAE y en Bucket Accuracy de negocio.

`DECISION = GO` solo si pasan el gate absoluto **y** el competitivo.

In [24]:
# =============================================================================
# SECCION 20 — GATE GO/NO-GO v1.9.0
# Combina el gate absoluto (16j original) con el benchmark vs constante (1b).
# Decision GO requiere pasar AMBOS.
# =============================================================================

print(f"\n{'='*95}")
print(f"  GATE GO/NO-GO v1.9.0 -- Modelo: {best_model_type_final}")
print(f"{'='*95}")

# --- 20.1 Metricas del modelo (predictions_final, q=best_q_final) -----------
r_gate = compute_full_metrics(predictions_final, actual_days, mask)
ece_gate = expected_calibration_error(bp_final[mask], seq_test.y_bucket[mask])
pred_ev_gate = decode_expected_value(bp_final, CONFIG)
print(f"  Calculando C-index (EV decoder, muestra 50K)...")
cindex_gate = concordance_index(pred_ev_gate, actual_days, mask)

# --- 20.2 GATE ABSOLUTO ------------------------------------------------------
print(f"\n  {'-'*70}")
print(f"  GATE ABSOLUTO")
print(f"  {'-'*70}")
print(f"  {'Gate':<25} {'Valor':>10} {'Threshold':>12} {'Estado':>8}")
print(f"  {'-'*60}")

gates_abs = [
    ("MAE",          r_gate['mae'],        5.0,  '<'),
    ("p90",          r_gate['p90e'],       12.0, '<'),
    ("Late%",        r_gate['late%']*100,  30.0, '<='),
    ("EWR@7d",       r_gate['ewr7']*100,   40.0, '>='),
    ("ECE",          ece_gate,             0.10, '<'),
    ("C-index (EV)", cindex_gate,          0.70, '>'),
]
all_pass_abs = True
for label, val, thr, op in gates_abs:
    if   op == '<':  passed = val < thr;  thr_str = f"< {thr}"
    elif op == '<=': passed = val <= thr; thr_str = f"<= {thr}"
    elif op == '>=': passed = val >= thr; thr_str = f">= {thr}"
    else:            passed = val > thr;  thr_str = f"> {thr}"
    all_pass_abs = all_pass_abs and passed
    if label in ('Late%', 'EWR@7d'):       val_str = f"{val:.1f}%"
    elif label in ('ECE', 'C-index (EV)'): val_str = f"{val:.4f}"
    else:                                  val_str = f"{val:.2f}d"
    print(f"  {label:<25} {val_str:>10} {thr_str:>12} {('PASA' if passed else 'FALLA'):>8}")

# --- 20.3 GATE COMPETITIVO vs PREDICTOR CONSTANTE ---------------------------
# Bucketizacion de negocio (4 clases) — definida AQUI, una sola vez.
def _business_bucket(days_arr):
    """4 buckets de negocio: [0,3) urgente, [3,7) semana, [7,14), [14+)."""
    d = np.asarray(days_arr, dtype=np.float64)
    return np.where(d < 3, 0, np.where(d < 7, 1, np.where(d < 14, 2, 3)))

naive_pred_val = float(np.median(actual_days[mask]))
naive_pred     = np.full(len(predictions_final), naive_pred_val)
r_naive        = compute_full_metrics(naive_pred, actual_days, mask)

model_ba = float((_business_bucket(predictions_final[mask]) ==
                  _business_bucket(actual_days[mask])).mean())
naive_ba = float((_business_bucket(naive_pred[mask]) ==
                  _business_bucket(actual_days[mask])).mean())

print(f"\n  {'-'*70}")
print(f"  GATE COMPETITIVO -- Modelo vs Constante (pred={naive_pred_val:.1f}d)")
print(f"  {'-'*70}")
print(f"  {'Metrica':<20} {'Modelo':>10} {'Constante':>11} {'Gana?':>8}")
print(f"  {'-'*52}")

comp_mae = r_gate['mae'] < r_naive['mae']
comp_ba  = model_ba > naive_ba
print(f"  {'MAE':<20} {r_gate['mae']:>9.2f}d {r_naive['mae']:>10.2f}d "
      f"{('SI' if comp_mae else 'NO'):>8}")
print(f"  {'Bucket Accuracy':<20} {model_ba*100:>9.1f}% {naive_ba*100:>10.1f}% "
      f"{('SI' if comp_ba else 'NO'):>8}")

all_pass_comp = comp_mae and comp_ba
print(f"\n  Gate competitivo: {'PASA' if all_pass_comp else 'FALLA'}")
if not all_pass_comp:
    print(f"  -> El modelo NO le gana a una constante. No es shippable como")
    print(f"     timing engine, sin importar los thresholds absolutos.")

# --- 20.4 DECISION FINAL -----------------------------------------------------
final_go = all_pass_abs and all_pass_comp
print(f"\n{'='*95}")
if final_go:
    print(f"  DECISION GO/NO-GO v1.9.0: GO")
else:
    motivo = []
    if not all_pass_abs:  motivo.append('gate absoluto')
    if not all_pass_comp: motivo.append('gate competitivo')
    print(f"  DECISION GO/NO-GO v1.9.0: NO-GO ({' + '.join(motivo)} FALLA)")
print(f"{'='*95}")

# --- Persistir gate ----------------------------------------------------------
gate_summary = {
    'version': 'v1.9.0',
    'model': best_model_type_final,
    'decoder_timing': f'q={best_q_final:.2f}',
    'mae_d': float(r_gate['mae']),
    'p90_d': float(r_gate['p90e']),
    'late_pct': float(r_gate['late%']),
    'ewr7d': float(r_gate['ewr7']),
    'ece': float(ece_gate),
    'cindex_ev': float(cindex_gate),
    'naive_pred_d': naive_pred_val,
    'naive_mae_d': float(r_naive['mae']),
    'model_bucket_acc': model_ba,
    'naive_bucket_acc': naive_ba,
    'gate_absolute_pass': bool(all_pass_abs),
    'gate_competitive_pass': bool(all_pass_comp),
    'final_decision': 'GO' if final_go else 'NO-GO',
    'abstention_policy': abstention_policy,
    'abstention_threshold': float(abstention_threshold),
}
_gate_path = OUTPUT_DIR / 'gate_final_v1_9_0.json'
with open(_gate_path, 'w') as _f:
    json.dump(gate_summary, _f, indent=2)
print(f"\n  Gate persistido: {_gate_path}")



  GATE GO/NO-GO v1.9.0 -- Modelo: TCN_CAL
  Calculando C-index (EV decoder, muestra 50K)...

  ----------------------------------------------------------------------
  GATE ABSOLUTO
  ----------------------------------------------------------------------
  Gate                           Valor    Threshold   Estado
  ------------------------------------------------------------
  MAE                           10.48d        < 5.0    FALLA
  p90                           26.88d       < 12.0    FALLA
  Late%                           4.8%      <= 30.0     PASA
  EWR@7d                         36.0%      >= 40.0    FALLA
  ECE                           0.1174        < 0.1    FALLA
  C-index (EV)                  0.7413        > 0.7     PASA

  ----------------------------------------------------------------------
  GATE COMPETITIVO -- Modelo vs Constante (pred=6.0d)
  ----------------------------------------------------------------------
  Metrica                  Modelo   Constante    Gana

## 21. Action Cards

Scoring accionable para CRM. Usa la `abstention_mask` unica de la Seccion 19.

**Advertencia (conservada del original):** este scoring corre sobre el TEST SET offline. No es inferencia productiva per-user. Es DEMO/TEMPLATE de la logica de segmentacion.

In [25]:
# ═══════════════════════════════════════════════════════════════════════════════
# ACTION CARDS v1.5.7 — Scoring accionable para CRM
#
# ⚠️  ADVERTENCIA IMPORTANTE (v1.5.7):
# Este scoring se ejecuta sobre el TEST SET OFFLINE.
# NO es una corrida de inferencia productiva per-user.
# Los conteos y distribuciones reflejan la composición del test set
# (incluyendo ~20% censurados que no tendrán cobro en la ventana).
#
# Para producción:
#   1. Usar score_censoring_policy(X_new) de 16g sobre datos frescos
#   2. Correr predictions sobre secuencias de inferencia reales
#   3. NO usar estos conteos como forecast CRM directo
#
# Este bloque se conserva como DEMO / TEMPLATE de la lógica de Action Cards.
# ═══════════════════════════════════════════════════════════════════════════════

# --- WARNING BANNER ---
print(f"\n" + "WARNING "*10)
print(f"  ADVERTENCIA: Este scoring corre sobre TEST SET OFFLINE.")
print(f"  NO es inferencia productiva. Usar como TEMPLATE/DEMO solamente.")
print(f"WARNING "*10 + "\n")

# --- PASO 1: Predicciones del modelo final ---
predicted_days = predictions_final

# --- PASO 2: Abstention via P(censored) del clasificador LogReg ---
# abstention_mask viene de la Seccion 19 (politica unificada v1.9.0).
# Ya no se reconstruye aqui: una sola fuente, sin try/except fragil.
assert 'abstention_mask' in dir(), 'Ejecutar la Seccion 19 antes de Action Cards'

# --- PASO 3: Asignar acciones ---
actions = np.full(len(predicted_days), 'no_action', dtype=object)

# Regla 1: Si abstención → no_action (usuario probablemente no será cobrado)
# Regla 2: Si no abstención:
#   - predicted_days <= 7  → send_reminder (urgente)
#   - predicted_days <= 14 → send_reminder (anticipado)
#   - predicted_days <= 30 → monitor
#   - predicted_days > 30  → no_action

active_mask = ~abstention_mask
actions[active_mask & (predicted_days <= 14)] = 'send_reminder'
actions[active_mask & (predicted_days > 14) & (predicted_days <= 30)] = 'monitor'
# active_mask & predicted_days > 30 stays as no_action

# --- PASO 4: Segmentos detallados ---
segments = np.where(predicted_days <= 7, '≤7d',
           np.where(predicted_days <= 14, '8-14d',
           np.where(predicted_days <= 30, '15-30d', '>30d/no-event')))

print(f"\n{'='*80}")
print("ACTION CARDS v1.5.7 — Scoring Accionable")
print(f"{'='*80}")

# Distribución de acciones
from collections import Counter
action_counts = Counter(actions)
print(f"\n  Distribución de acciones:")
for act, cnt in sorted(action_counts.items()):
    pct = cnt / len(actions) * 100
    print(f"    {act:<20}: {cnt:>8,} ({pct:.1f}%)")

print(f"\n  Total send_reminder: {action_counts.get('send_reminder', 0):,}")
print(f"  Total monitor:       {action_counts.get('monitor', 0):,}")
print(f"  Total no_action:     {action_counts.get('no_action', 0):,}")
print(f"  Abstenciones:        {abstention_mask.sum():,} ({abstention_mask.mean()*100:.1f}%)")

# --- PASO 5: Action Cards por segmento ---
action_cards = []

for seg_key, seg_config in CONFIG['action_segments'].items():
    seg_label = seg_config['label']
    mask_seg = segments == seg_label
    n_users = mask_seg.sum()
    
    if n_users == 0:
        continue
    
    pct = n_users / len(segments) * 100
    median_days = np.median(predicted_days[mask_seg])
    n_send = int((actions[mask_seg] == 'send_reminder').sum())
    n_monitor = int((actions[mask_seg] == 'monitor').sum())
    n_noaction = int((actions[mask_seg] == 'no_action').sum())
    
    if seg_key == 'immediate':
        decision = "send_reminder: Push/SMS T-1 para fondeo urgente"
        kpi = "Approval rate +5-10%, retries -15%"
    elif seg_key == 'short_term':
        decision = "send_reminder: Push T-2/T-3 + análisis de liquidez"
        kpi = "Approval rate +3-7%, quejas -5%"
    elif seg_key == 'medium_term':
        decision = "monitor: Campaña educativa, reminder T-7"
        kpi = "Activación mensual +2-5%"
    else:
        decision = "no_action: Monitoreo pasivo"
        kpi = "Prevención de churn -2-3%"
    
    card = {
        'segment': seg_label,
        'n_users': int(n_users),
        'pct_total': float(pct),
        'predicted_days_median': float(median_days),
        'n_send_reminder': n_send,
        'n_monitor': n_monitor,
        'n_no_action': n_noaction,
        'decision': decision,
        'kpi_impact': kpi,
    }
    action_cards.append(card)
    
    print(f"\n📋 Segmento: {seg_label}")
    print(f"   N: {n_users:,} ({pct:.1f}%)")
    print(f"   Días predichos (median): {median_days:.1f}")
    print(f"   Acciones: send={n_send:,} | monitor={n_monitor:,} | no_action={n_noaction:,}")
    print(f"   Decisión: {decision}")
    print(f"   KPI: {kpi}")

# Save
with open(OUTPUT_DIR / 'action_cards_v1_5_6.json', 'w') as f:
    json.dump(action_cards, f, indent=2)

print(f"\n✅ Action Cards v1.5.7 guardadas: action_cards_v1_5_6.json")
print(f"   send_reminder > 0: {'✅ ACCIONABLE' if action_counts.get('send_reminder', 0) > 0 else '⚠️  REVISAR THRESHOLDS'}")



WARNING WARNING WARNING WARNING WARNING WARNING WARNING WARNING WARNING WARNING 
  ADVERTENCIA: Este scoring corre sobre TEST SET OFFLINE.
  NO es inferencia productiva. Usar como TEMPLATE/DEMO solamente.
WARNING WARNING WARNING WARNING WARNING WARNING WARNING WARNING WARNING WARNING 


ACTION CARDS v1.5.7 — Scoring Accionable

  Distribución de acciones:
    monitor             :       88 (0.0%)
    no_action           :   26,973 (2.9%)
    send_reminder       :  893,738 (97.1%)

  Total send_reminder: 893,738
  Total monitor:       88
  Total no_action:     26,973
  Abstenciones:        26,973 (2.9%)

📋 Segmento: ≤7d
   N: 804,870 (87.4%)
   Días predichos (median): 1.5
   Acciones: send=777,899 | monitor=0 | no_action=26,971
   Decisión: send_reminder: Push/SMS T-1 para fondeo urgente
   KPI: Approval rate +5-10%, retries -15%

📋 Segmento: 8-14d
   N: 115,840 (12.6%)
   Días predichos (median): 8.8
   Acciones: send=115,839 | monitor=0 | no_action=1
   Decisión: send_reminder: Push

## 22. Bundle de Artefactos de Inferencia + Checklist de Reproducibilidad

Consolida todo lo necesario para inferencia de kernel fresco y verifica que cada artefacto exista en disco.

In [26]:
# SECCION 22 — Bundle de inferencia + checklist (ADD v1.9.0)

# Manifiesto unico de inferencia: que modelo, que decoder, que features,
# que politica de abstencion. Junto con los .joblib/.pkl persistidos, esto
# permite reconstruir el scoring sin re-entrenar.
inference_bundle = {
    'notebook_version': 'v1.9.0',
    'best_model_type_final': best_model_type_final,
    'best_q_final': float(best_q_final),
    'hmm_features': HMM_FEATURES,
    'bucket_edges': CONFIG['bucket_edges'],
    'bucket_centers': CONFIG['bucket_centers'],
    'clip_zscore': CONFIG['clip_zscore'],
    'lookback_window': CONFIG['lookback_window'],
    'temperatures': {k: float(v) for k, v in temperatures.items()},
    'abstention_policy': abstention_policy,
    'abstention_threshold': float(abstention_threshold),
}
_bundle_path = OUTPUT_DIR / 'inference_bundle_v1_9_0.json'
with open(_bundle_path, 'w') as _f:
    json.dump(inference_bundle, _f, indent=2)
print(f'  inference_bundle persistido: {_bundle_path}')

# Checklist: todo lo que la inferencia de kernel fresco necesita.
print(f"\n{'='*70}")
print('  CHECKLIST DE ARTEFACTOS DE INFERENCIA')
print(f"{'='*70}")
required = [
    'scaler_main.joblib',
    'feature_manifest.json',
    'temperatures.joblib',
    'final_selection_v1_9_0.json',
    'inference_bundle_v1_9_0.json',
    'censoring_classifier_v1_9_0.pkl',
    'gate_final_v1_9_0.json',
]
for mt in CONFIG['models_to_train']:
    required.append(f'{mt}_model_v1_9_0.pt')

all_ok = True
for a in required:
    exists = (OUTPUT_DIR / a).exists()
    all_ok = all_ok and exists
    print(f"  [{'OK ' if exists else 'FALTA'}] {a}")

print(f"\n  {'Todos los artefactos presentes.' if all_ok else 'FALTAN artefactos — revisar arriba.'}")


  inference_bundle persistido: artifacts_v1_9_0/inference_bundle_v1_9_0.json

  CHECKLIST DE ARTEFACTOS DE INFERENCIA
  [OK ] scaler_main.joblib
  [OK ] feature_manifest.json
  [OK ] temperatures.joblib
  [OK ] final_selection_v1_9_0.json
  [OK ] inference_bundle_v1_9_0.json
  [OK ] censoring_classifier_v1_9_0.pkl
  [OK ] gate_final_v1_9_0.json
  [OK ] GRU_model_v1_9_0.pt
  [OK ] RNN_model_v1_9_0.pt
  [OK ] LSTM_model_v1_9_0.pt
  [OK ] TCN_model_v1_9_0.pt
  [OK ] DEEP_GRU_model_v1_9_0.pt

  Todos los artefactos presentes.
